# Step5 Eigenspace 对比：Mat32 / eps / top_q

目标：
- 复用 `v1_mat32_eps_topq_sanity.ipynb` 的数据与参数
- 在步骤 5（估计矩阵 A 的 top eigenspace）比较不同方法的耗时与残差
- 基于不同 eigenspace 再跑完整算法（预条件 + PCG），检查准确性与稳定性

说明：
- 结果只打印在 notebook 中，不写 CSV
- 计算量很大，请确认内存与 CPU 资源

Result:
Surrogate Method, replacing original A with a coarser/looser/subsample can significantly increase the speed of estimation of eigenspace without loosing precision. And current estimation method is good enough.

## Prepare

In [12]:
'''
## For github import
import os
import sys

GITHUB_USER = "Yifiwifi"
REPO_NAME = "EFGP-Eigenpro"
SUB_DIR = "efgp_eigenpro_py"
PROJECT_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(PROJECT_PATH):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
else:
    %cd {PROJECT_PATH}
    !git pull origin main

CODE_ROOT = os.path.join(PROJECT_PATH, SUB_DIR)
if CODE_ROOT not in sys.path:
    sys.path.append(CODE_ROOT)

print("Checking runtime dependencies")
!pip install cufinufft cupy-cuda12x --extra-index-url https://pypi.nvidia.com

requirements_path = os.path.join(CODE_ROOT, "requirements.txt")
if os.path.exists(requirements_path):
    !pip install -r {requirements_path}

sanity_check_path = os.path.join(CODE_ROOT, "gpu/sanity_check")
if os.path.exists(sanity_check_path):
    os.chdir(sanity_check_path)
    print("cwd:", os.getcwd())
else:
    print("sanity_check path not found:", sanity_check_path)

# Refresh runtime library path for some Colab images
os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
!ldconfig /usr/local/lib

print("=" * 40)
try:
    import torch
    import cupy as cp
    import cufinufft
    cp.cuda.Stream.null.synchronize()
    print("PyTorch:", torch.__version__)
    print("GPU:", torch.cuda.get_device_name(0))
    print("cufinufft import ok")
except Exception as e:
    print("runtime check failed:", e)
print("=" * 40)
'''


'\n## For github import\nimport os\nimport sys\n\nGITHUB_USER = "Yifiwifi"\nREPO_NAME = "EFGP-Eigenpro"\nSUB_DIR = "efgp_eigenpro_py"\nPROJECT_PATH = f"/content/{REPO_NAME}"\n\nif not os.path.exists(PROJECT_PATH):\n    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git\nelse:\n    %cd {PROJECT_PATH}\n    !git pull origin main\n\nCODE_ROOT = os.path.join(PROJECT_PATH, SUB_DIR)\nif CODE_ROOT not in sys.path:\n    sys.path.append(CODE_ROOT)\n\nprint("Checking runtime dependencies")\n!pip install cufinufft cupy-cuda12x --extra-index-url https://pypi.nvidia.com\n\nrequirements_path = os.path.join(CODE_ROOT, "requirements.txt")\nif os.path.exists(requirements_path):\n    !pip install -r {requirements_path}\n\nsanity_check_path = os.path.join(CODE_ROOT, "gpu/sanity_check")\nif os.path.exists(sanity_check_path):\n    os.chdir(sanity_check_path)\n    print("cwd:", os.getcwd())\nelse:\n    print("sanity_check path not found:", sanity_check_path)\n\n# Refresh runtime library path for som

In [13]:
import os
import sys
import time
import traceback
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

# Ensure project root is importable no matter where notebook kernel starts.
_here = Path.cwd().resolve()
_candidates = [
    _here,
    _here.parent,
    _here.parent.parent,
    _here.parent.parent.parent,
    Path("D:/NU/ML"),
]
for p in _candidates:
    pkg_dir = p / "efgp_eigenpro_py"
    if pkg_dir.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break

from efgp_eigenpro_py.kernels import make_matern
from efgp_eigenpro_py.efgp_solver import EFGPSolver
from efgp_eigenpro_py.discretization import GridSpec
from efgp_eigenpro_py.benchmark import (
    make_dataset,
    make_test_set,
    true_func_2d,
    compute_rmse,
)
from efgp_eigenpro_py.gpu.backends import BackendConfig, build_gpu_backend_bundle
from efgp_eigenpro_py.gpu.contexts import GPUOperatorContext, ensure_gpu_data_context
from efgp_eigenpro_py.gpu.iterative_solvers import pcg_solve_gpu
from efgp_eigenpro_py.gpu.v1_ops import (
    apply_A_v1,
    predict_v1,
    solve_beta_plain_cg_v1,
)
from efgp_eigenpro_py.gpu.v2_preconditioner import GPUPreconditionerData, apply_preconditioner_v2
from efgp_eigenpro_py.gpu.surrogate_ops import (
    EigenspaceConfig,
    estimate_top_eigenspace_eigenpro_nystrom,
    estimate_top_eigenspace_v3,
    gpu_precompute_v1,
    mu_for_precond_from_eig,
)
from efgp_eigenpro_py.gpu.versions import GPURunConfig

np.set_printoptions(precision=6, suppress=True)
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])

cwd: d:\NU\ML\efgp_eigenpro_py\gpu\sanity_check
sys.path[0]: D:\NU\ML


In [14]:
# ----- user requested test matrix -----
REG_LAMBDA = 0.1
N_TRAIN = 1_00_000
DIM = 2
LENGTHSCALE = 0.1
NU = 1.5  # Mat32
EPS_LIST = [1e-6]
TOP_Q_LIST = [80]

N_TEST = 2_000
NOISE = 0.02
SEED_TRAIN = 20230
SEED_TEST = 2027

SOLVE_TOL = 1e-6
GPU_TOL = SOLVE_TOL
GPU_MAXITER = 3000
GPU_CHUNK_SIZE = None
GPU_DEBUG_FINITE_CHECKS = False
GPU_NUFFT = "auto"
L2_SCALED = True
OVERSAMPLE_DEFAULT = 16
BLOCK_MATVEC_COLS = 16
BLOCK_MATVEC_MIN_COLS = 4

# EigenPro-style Nyström top-q on W0=K0[S,S], then lift + optional Rayleigh–Ritz (v3_eigenspace)
EIGENPRO_NYSTROM_SURROGATE_SIZE = 500  # ~10 * (top_q+1) when top_q=64
EIGENPRO_NYSTROM_LOWFREQ_RATIO = 0.25
EIGENPRO_NYSTROM_OVERSAMPLE = 10
EIGENPRO_NYSTROM_RITZ_REFINE = False
EIGENPRO_NYSTROM_SEED = 0
EIGENPRO_NYSTROM_BLOCK_ROWS = 8192  # None => auto (see v3 _auto_block_rows); or set e.g. 8192
EIGENPRO_NYSTROM_RITZ_BLOCK_COLS = 16
EIGENPRO_NYSTROM_LIFT = False  # False: coordinate-only I[:,S]V, no K[:,S]T^-1 full lift (profiling)

BASELINE_EIG_METHOD = "subspace_iter"
# 本 notebook 仅对比三种 eigenspace：subspace_iter（默认）、eigenpro_nystrom、cupy_eigsh（走 CUSTOM_EIGSH_CFG）
ENABLE_CUSTOM_EIGENSPACE = True
CUSTOM_METHODS_ENABLED = ["cupy_eigsh"]

CUSTOM_EIGSH_CFG = {
    "tol": 1e-6,
    "maxiter": 200,
    "ncv": None,
    "which": "LA",
    "warm_start_strategy": "none",  # none | random | power1 | init_q_first_col
}
CUSTOM_LOBPCG_REFINE_CFG = {
    "tol": 1e-6,
    "maxiter": 40,
    "block_size": None,
    "oversample": OVERSAMPLE_DEFAULT,
    "min_oversample": 16,
    "init_source": "rand_subspace_rr",
    "init_power_iters": 0,
    "fallback_maxiter": 3,
    "use_precond": True,
    "precond": None,
    "precond_block": None,
}
CUSTOM_RAND_SUBSPACE_RR_CFG = {
    "tol": 1e-6,
    "maxiter": 3,
    "block_size": None,
    "oversample": OVERSAMPLE_DEFAULT,
    "min_oversample": 16,
    "rr_check_every": 1,
    "residual_check_every": 1,
    "rr_warmup_iters": 1,
    "residual_q": 0.90,
    "residual_max_factor": 5.0,
}
CUSTOM_MIXED_SUBSPACE_RR_CFG = {
    "tol": 1e-6,
    "maxiter": 20,
    "block_size": None,
    "oversample": OVERSAMPLE_DEFAULT,
    "min_oversample": 16,
    "lowp_dtype": "complex64",
    "matvec_dtype": "complex64",
    "matvec_block_highp": None,
    "matvec_block_lowp": None,
    "allow_implicit_lowp": False,
    "rr_check_every": 4,
    "residual_check_every": 4,
    "rr_warmup_iters": 4,
    "reorth_every": 4,
    "residual_q": 0.90,
    "residual_max_factor": 5.0,
}
CUSTOM_POLY_FILTERED_SUBSPACE_CFG = {
    "tol": 1e-6,
    "maxiter": 10,
    "block_size": None,
    "oversample": OVERSAMPLE_DEFAULT,
    "min_oversample": 16,
    "poly_degree": None,        # None => adaptive; set int to override
    "pilot_iters": 1,
    "pilot_block": 48,
    "stopband_min": None,       # None => adaptive from pilot lam_min
    "cutoff_update_every": 2,
    "cutoff_warmup_iters": 2,
    "rr_check_every": 2,
    "residual_check_every": 2,
    "rr_warmup_iters": 2,
    "residual_q": 0.90,
    "residual_max_factor": 5.0,
}
CUSTOM_MIXED_POLY_FILTERED_SUBSPACE_CFG = {
    "tol": 1e-6,
    "maxiter": 10,
    "block_size": None,
    "oversample": OVERSAMPLE_DEFAULT,
    "min_oversample": 16,
    "poly_degree": None,        # None => adaptive; set int to override
    "lowp_dtype": "complex64",
    "matvec_dtype": "complex64",
    "matvec_block_highp": None,
    "matvec_block_lowp": None,
    "allow_implicit_lowp": False,
    "pilot_iters": 1,
    "pilot_block": 48,
    "stopband_min": None,       # None => adaptive from pilot lam_min
    "cutoff_update_every": 2,
    "cutoff_warmup_iters": 2,
    "rr_check_every": 2,
    "residual_check_every": 2,
    "rr_warmup_iters": 2,
    "reorth_every": 4,
    "residual_q": 0.90,
    "residual_max_factor": 5.0,
}
COMBO_GRID_VALUES = {
    "grid_scale": [0.1], #0.1
    "eps_factor": [ 100.0],
    "subsample_frac": [0.3],
}
COMBO_COMMON_CFG = {
    "subsample_seed": 0,
    "sur_iter": 1,
    "refine_iter": 1,
    "oversample": OVERSAMPLE_DEFAULT,
    "block_size": None,
}
COMBO_ENABLED = False  # 关闭 combo/surrogate 网格扫描，只保留上方 EIG_METHODS + 自定义 cupy_eigsh

SURROGATE_ENABLED = None
# "coarse_grid", "loose_eps", "subsample"
SURROGATE_METHODS = {
    "coarse_grid": {
        "grid_scale": 0.35,
        "grid_m": None,
        "sur_iter": 1,
        "refine_iter": 1,
        "oversample": OVERSAMPLE_DEFAULT,
        "block_size": None,
    },
    "loose_eps": {
        "eps_factor": 50.0,
        "sur_iter": 1,
        "refine_iter": 1,
        "oversample": OVERSAMPLE_DEFAULT,
        "block_size": None,
    },
    "subsample": {
        "subsample_frac": 0.05,
        "subsample_seed": 0,
        "sur_iter": 1,
        "refine_iter": 1,
        "oversample": OVERSAMPLE_DEFAULT,
        "block_size": None,
    },
}


def _combo_tag(val: float) -> str:
    return str(val).replace(".", "p")


COMBO_VARIANTS = []
for grid_scale in COMBO_GRID_VALUES["grid_scale"]:
    for eps_factor in COMBO_GRID_VALUES["eps_factor"]:
        for subsample_frac in COMBO_GRID_VALUES["subsample_frac"]:
            combo_name = (
                f"combo_g{_combo_tag(grid_scale)}"
                f"_e{_combo_tag(eps_factor)}"
                f"_s{_combo_tag(subsample_frac)}"
            )
            combo_cfg = {
                "name": combo_name,
                "kind": "combo",
                "grid_scale": grid_scale,
                "grid_m": None,
                "eps_factor": eps_factor,
                "subsample_frac": subsample_frac,
                **COMBO_COMMON_CFG,
            }
            COMBO_VARIANTS.append(combo_cfg)

EIG_METHODS = [
    {
        "name": "subspace_iter",    # actually use, v3_subspace_iter
        "method": "v3_subspace_iter",
        "n_iter": 3,
        "block_size": None,
        "oversample": OVERSAMPLE_DEFAULT,
        "callable": None,
    },
    {
        "name": "eigenpro_nystrom",
        "method": "v3_eigenpro_nystrom",
        "n_iter": 0,
        "block_size": None,
        "oversample": EIGENPRO_NYSTROM_OVERSAMPLE,
        "surrogate_size": EIGENPRO_NYSTROM_SURROGATE_SIZE,
        "surrogate_lowfreq_ratio": EIGENPRO_NYSTROM_LOWFREQ_RATIO,
        "surrogate_ritz_refine": EIGENPRO_NYSTROM_RITZ_REFINE,
        "surrogate_seed": EIGENPRO_NYSTROM_SEED,
        "surrogate_block_rows": EIGENPRO_NYSTROM_BLOCK_ROWS,
        "surrogate_ritz_block_cols": EIGENPRO_NYSTROM_RITZ_BLOCK_COLS,
        "surrogate_lift": EIGENPRO_NYSTROM_LIFT,
        "callable": None,
    },
]

if SURROGATE_ENABLED is not None:
    for sur_name in SURROGATE_ENABLED:
        sur_cfg = SURROGATE_METHODS.get(sur_name)
        if sur_cfg is None:
            continue
        EIG_METHODS.append(
            {
                "name": f"sur_{sur_name}",
                "method": "surrogate",
                "surrogate": {"name": sur_name, **sur_cfg},
                "callable": None,
            }
        )

if COMBO_ENABLED:
    allowed = None
    if isinstance(COMBO_ENABLED, (list, tuple, set)):
        allowed = {name for name in COMBO_ENABLED}
    for combo_cfg in COMBO_VARIANTS:
        if allowed is not None and combo_cfg["name"] not in allowed:
            continue
        EIG_METHODS.append(
            {
                "name": f"sur_{combo_cfg['name']}",
                "method": "surrogate",
                "surrogate": combo_cfg,
                "callable": None,
            }
        )

RUN_FULL_SOLVE = True

print("Run mode: print-only (no CSV output)")
print("EPS_LIST:", EPS_LIST)
print("TOP_Q_LIST:", TOP_Q_LIST)
print("EIG_METHODS:", [cfg["name"] for cfg in EIG_METHODS])
print("ENABLE_CUSTOM_EIGENSPACE:", ENABLE_CUSTOM_EIGENSPACE)
print("CUSTOM_METHODS_ENABLED:", CUSTOM_METHODS_ENABLED)
print("CUSTOM_EIGSH_CFG:", CUSTOM_EIGSH_CFG)
print("CUSTOM_LOBPCG_REFINE_CFG:", CUSTOM_LOBPCG_REFINE_CFG)
print("CUSTOM_RAND_SUBSPACE_RR_CFG:", CUSTOM_RAND_SUBSPACE_RR_CFG)
print("CUSTOM_MIXED_SUBSPACE_RR_CFG:", CUSTOM_MIXED_SUBSPACE_RR_CFG)
print("CUSTOM_POLY_FILTERED_SUBSPACE_CFG:", CUSTOM_POLY_FILTERED_SUBSPACE_CFG)
print("CUSTOM_MIXED_POLY_FILTERED_SUBSPACE_CFG:", CUSTOM_MIXED_POLY_FILTERED_SUBSPACE_CFG)
print("SURROGATE_ENABLED:", SURROGATE_ENABLED)
print("COMBO_ENABLED:", COMBO_ENABLED)
print("COMBO_GRID_VALUES:", COMBO_GRID_VALUES)
print("COMBO_COMMON_CFG:", COMBO_COMMON_CFG)
print("COMBO_VARIANTS:", len(COMBO_VARIANTS))
print("SURROGATE_METHODS:", SURROGATE_METHODS)
print("BASELINE_EIG_METHOD:", BASELINE_EIG_METHOD)
print("GPU_NUFFT:", GPU_NUFFT)
print("BLOCK_MATVEC_COLS:", BLOCK_MATVEC_COLS)
print("BLOCK_MATVEC_MIN_COLS:", BLOCK_MATVEC_MIN_COLS)
print("EIGENPRO_NYSTROM_SURROGATE_SIZE:", EIGENPRO_NYSTROM_SURROGATE_SIZE)
print("RUN_FULL_SOLVE:", RUN_FULL_SOLVE)

Run mode: print-only (no CSV output)
EPS_LIST: [1e-06]
TOP_Q_LIST: [80]
EIG_METHODS: ['subspace_iter', 'eigenpro_nystrom']
ENABLE_CUSTOM_EIGENSPACE: True
CUSTOM_METHODS_ENABLED: ['cupy_eigsh']
CUSTOM_EIGSH_CFG: {'tol': 1e-06, 'maxiter': 200, 'ncv': None, 'which': 'LA', 'warm_start_strategy': 'none'}
CUSTOM_LOBPCG_REFINE_CFG: {'tol': 1e-06, 'maxiter': 40, 'block_size': None, 'oversample': 16, 'min_oversample': 16, 'init_source': 'rand_subspace_rr', 'init_power_iters': 0, 'fallback_maxiter': 3, 'use_precond': True, 'precond': None, 'precond_block': None}
CUSTOM_RAND_SUBSPACE_RR_CFG: {'tol': 1e-06, 'maxiter': 3, 'block_size': None, 'oversample': 16, 'min_oversample': 16, 'rr_check_every': 1, 'residual_check_every': 1, 'rr_warmup_iters': 1, 'residual_q': 0.9, 'residual_max_factor': 5.0}
CUSTOM_MIXED_SUBSPACE_RR_CFG: {'tol': 1e-06, 'maxiter': 20, 'block_size': None, 'oversample': 16, 'min_oversample': 16, 'lowp_dtype': 'complex64', 'matvec_dtype': 'complex64', 'matvec_block_highp': None, 'm

In [15]:
# 数据准备（请确认机器内存/CPU 资源足够）
print("Building dataset ...")
x_train, y_train = make_dataset(DIM, N_TRAIN, true_func_2d, noise=NOISE, seed=SEED_TRAIN)
x_test, y_test = make_test_set(DIM, N_TEST, true_func_2d, seed=SEED_TEST)

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test :", x_test.shape, x_test.dtype)

kernel = make_matern(lengthscale=LENGTHSCALE, nu=NU, dim=DIM, variance=1.0)
print(kernel)

Building dataset ...
x_train: (100000, 2) float64
y_train: (100000,) float64
x_test : (2000, 2) float64
KernelSpec(fam='matern', dim=2, lengthscale=0.1, variance=1.0, nu=1.5)


In [16]:
def _mse_mae(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    diff = a - b
    mse = float(np.mean(np.abs(diff) ** 2))
    mae = float(np.mean(np.abs(diff)))
    return mse, mae


def _device_to_numpy(xp, arr):
    if hasattr(xp, "asnumpy"):
        return np.asarray(xp.asnumpy(arr))
    if hasattr(arr, "get"):
        return np.asarray(arr.get())
    return np.asarray(arr)


def _resolve_block_size(top_q, cfg):
    block_size = cfg.get("block_size")
    if block_size is None:
        oversample = int(cfg.get("oversample", 2))
        block_size = max(top_q + oversample, top_q + 1)
    return int(block_size)


def _apply_matvec_block(matvec, matvec_block, V, xp):
    if matvec_block is not None:
        return matvec_block(V)
    cols = [matvec(V[:, i]) for i in range(V.shape[1])]
    return xp.stack(cols, axis=1)


def _randn(xp, seed, shape):
    try:
        rng = xp.random.RandomState(seed)
        return rng.standard_normal(shape)
    except Exception:
        return xp.random.standard_normal(shape)


def _sort_eigpairs(eigvals, eigvecs, xp):
    idx = xp.argsort(xp.real(eigvals))[::-1]
    return xp.real(eigvals[idx]), eigvecs[:, idx]


def _prepare_init_basis(xp, size, block, seed, dtype, init_Q=None):
    if init_Q is None:
        Q0 = _randn(xp, seed, (size, block)).astype(dtype)
    else:
        Q0 = xp.asarray(init_Q)
        if Q0.ndim == 1:
            Q0 = Q0.reshape(-1, 1)
        if int(Q0.shape[0]) != int(size):
            raise ValueError(f"init_Q has wrong row count: {Q0.shape[0]} != {size}")
        if int(Q0.shape[1]) < int(block):
            extra = _randn(xp, seed + 97, (size, int(block) - int(Q0.shape[1]))).astype(Q0.dtype)
            Q0 = xp.concatenate([Q0, extra], axis=1)
        elif int(Q0.shape[1]) > int(block):
            Q0 = Q0[:, : int(block)]
    Q64 = xp.asarray(Q0, dtype=xp.complex128)
    Q64, _ = xp.linalg.qr(Q64, mode="reduced")
    return xp.asarray(Q64[:, : int(block)], dtype=dtype)


def _orthonormalize_iterate_mixed(Y, xp, lowp_dtype, use_highp_qr):
    if use_highp_qr or lowp_dtype == xp.complex128:
        Yq = Y if getattr(Y, "dtype", None) == xp.complex128 else xp.asarray(Y, dtype=xp.complex128)
        Q64, _ = xp.linalg.qr(Yq, mode="reduced")
        return xp.asarray(Q64, dtype=lowp_dtype)
    Yq = Y if getattr(Y, "dtype", None) == lowp_dtype else xp.asarray(Y, dtype=lowp_dtype)
    try:
        Q, _ = xp.linalg.qr(Yq, mode="reduced")
        return xp.asarray(Q, dtype=lowp_dtype)
    except Exception:
        Y64 = xp.asarray(Y, dtype=xp.complex128)
        Q64, _ = xp.linalg.qr(Y64, mode="reduced")
        return xp.asarray(Q64, dtype=lowp_dtype)


# 自定义 eigenspace 方法移到下一单元。

def _normalize_eigpairs(result, xp):
    if isinstance(result, tuple):
        eigvals, eigvecs = result
    else:
        if not hasattr(result, "values") or not hasattr(result, "vectors"):
            raise ValueError("custom eigenspace must return values and vectors.")
        eigvals, eigvecs = result.values, result.vectors
    return SimpleNamespace(values=xp.asarray(eigvals), vectors=xp.asarray(eigvecs))


def _build_gpu_context(solver, x, y, cfg, grid_override=None):
    backend = build_gpu_backend_bundle(cfg.backend)
    data_ctx = ensure_gpu_data_context(backend, x, y, state=None)
    data_ctx.meta["debug_finite_checks"] = bool(cfg.debug_finite_checks)
    op_ctx = GPUOperatorContext()
    t0 = time.perf_counter()
    data_ctx = gpu_precompute_v1(
        backend,
        solver.kernel,
        solver.eps,
        solver.nufft_tol,
        data_ctx,
        op_ctx,
        l2scaled=solver.l2scaled,
        chunk_size=cfg.chunk_size,
        grid_override=grid_override,
    )
    t1 = time.perf_counter()
    return backend, data_ctx, op_ctx, float(t1 - t0)


def _apply_A_block_gpu(backend, data_ctx, op_ctx, reg_lambda, v_block):
    xp = backend.xp
    V = xp.asarray(v_block, dtype=xp.complex128)
    if V.ndim == 1:
        V = V.reshape(-1, 1)

    def _free_memory_pool():
        try:
            pool = xp.get_default_memory_pool()
            if pool is not None:
                pool.free_all_blocks()
        except Exception:
            pass

    def _apply_fast(Vc):
        w_flat = data_ctx.weights_gpu_flat
        gf = data_ctx.gf_gpu
        mtot = int(data_ctx.meta.get("mtot", 0))
        dim = int(data_ctx.meta.get("dim", 0))
        if w_flat is None or gf is None or mtot <= 0 or dim <= 0:
            raise RuntimeError("invalid precompute state")

        n, b = int(Vc.shape[0]), int(Vc.shape[1])
        if int(w_flat.size) != n:
            raise RuntimeError("vector size mismatch")

        shape = (mtot,) * dim
        pad_shape = tuple(int(s) for s in gf.shape) + (b,)
        WV = xp.multiply(w_flat.reshape(-1, 1), Vc).reshape(shape + (b,))
        pad = xp.zeros(pad_shape, dtype=gf.dtype)
        pad[tuple(slice(0, mtot) for _ in range(dim)) + (slice(None),)] = WV

        axes = tuple(range(dim))
        af = backend.fft.fftn(pad, axes=axes)
        gf_expand = xp.asarray(gf).reshape(tuple(int(s) for s in gf.shape) + (1,))
        vfft = backend.fft.ifftn(af * gf_expand, axes=axes)

        core_slice = tuple(slice(mtot - 1, 2 * mtot - 1) for _ in range(dim)) + (slice(None),)
        core = vfft[core_slice].reshape(n, b)
        out_chunk = xp.empty_like(Vc)
        xp.multiply(w_flat.reshape(-1, 1), core, out=out_chunk)
        out_chunk += float(reg_lambda) * Vc
        return out_chunk

    n, b = int(V.shape[0]), int(V.shape[1])
    max_cols = int(globals().get("BLOCK_MATVEC_COLS", 32) or 32)
    min_cols = int(globals().get("BLOCK_MATVEC_MIN_COLS", 8) or 8)
    max_cols = max(1, min(max_cols, b))
    min_cols = max(1, min(min_cols, max_cols))

    if b > max_cols:
        out_block = xp.empty_like(V)
        for j0 in range(0, b, max_cols):
            j1 = min(j0 + max_cols, b)
            out_block[:, j0:j1] = _apply_A_block_gpu(
                backend,
                data_ctx,
                op_ctx,
                reg_lambda,
                V[:, j0:j1],
            )
        return out_block

    try:
        return _apply_fast(V)
    except Exception as exc:
        oom_type = getattr(getattr(xp, "cuda", None), "memory", None)
        oom_type = getattr(oom_type, "OutOfMemoryError", MemoryError)
        if isinstance(exc, oom_type) and b > min_cols:
            _free_memory_pool()
            mid = max(min_cols, b // 2)
            if mid < b:
                out_block = xp.empty_like(V)
                out_block[:, :mid] = _apply_A_block_gpu(backend, data_ctx, op_ctx, reg_lambda, V[:, :mid])
                out_block[:, mid:] = _apply_A_block_gpu(backend, data_ctx, op_ctx, reg_lambda, V[:, mid:])
                return out_block
        _free_memory_pool()
        out_block = xp.empty_like(V)
        for i in range(V.shape[1]):
            apply_A_v1(
                backend,
                data_ctx,
                V[:, i],
                float(reg_lambda),
                op_ctx,
                out=out_block[:, i],
            )
        return out_block


def _build_surrogate_grid(fine_m, fine_h, grid_scale=None, grid_m=None):
    if grid_m is not None:
        m_coarse = int(grid_m)
    else:
        if grid_scale is None:
            grid_scale = 1.0
        m_coarse = max(1, int(fine_m * float(grid_scale)))
    xis = np.arange(-m_coarse, m_coarse + 1) * float(fine_h)
    return GridSpec(xis=xis, h=float(fine_h), mtot=xis.size, hm=m_coarse)


def _embed_coarse_to_fine(U_sur, mtot_sur, mtot_fine, dim, xp):
    if int(mtot_sur) == int(mtot_fine):
        return U_sur
    if mtot_sur > mtot_fine:
        raise ValueError("surrogate grid is larger than fine grid")
    start = (int(mtot_fine) - int(mtot_sur)) // 2
    if start < 0:
        raise ValueError("invalid surrogate/fine grid alignment")

    shape_sur = (int(mtot_sur),) * int(dim)
    shape_fine = (int(mtot_fine),) * int(dim)
    k = int(U_sur.shape[1])
    out = xp.zeros((int(mtot_fine) ** int(dim), k), dtype=U_sur.dtype)
    slicer = tuple(slice(start, start + int(mtot_sur)) for _ in range(int(dim)))

    for j in range(k):
        u_sur = U_sur[:, j].reshape(shape_sur)
        u_fine = xp.zeros(shape_fine, dtype=U_sur.dtype)
        u_fine[slicer] = u_sur
        out[:, j] = u_fine.reshape(-1)
    return out


def _build_surrogate_context(solver, x_full, y_full, gpu_cfg, sur_cfg, fine_meta):
    sur_tag = sur_cfg.get("name", "")
    sur_kind = sur_cfg.get("kind", sur_tag)
    x_use = x_full
    y_use = y_full
    grid_override = None
    eps_sur = float(solver.eps)

    if sur_kind in ("subsample", "combo"):
        frac = float(sur_cfg.get("subsample_frac", 1.0))
        seed = int(sur_cfg.get("subsample_seed", 0))
        n = int(x_full.shape[0])
        if frac < 1.0:
            rng = np.random.default_rng(seed)
            n_keep = max(1, int(n * frac))
            idx = rng.choice(n, size=n_keep, replace=False)
            x_use = x_full[idx]
            y_use = y_full[idx]
    if sur_kind in ("loose_eps", "combo"):
        eps_factor = float(sur_cfg.get("eps_factor", 1.0))
        eps_sur = float(solver.eps) * eps_factor
    if sur_kind in ("coarse_grid", "combo"):
        fine_mtot = int(fine_meta.get("mtot", 0))
        fine_m = int((fine_mtot - 1) // 2)
        fine_h = float(fine_meta.get("h", 0.0))
        grid_scale = sur_cfg.get("grid_scale")
        grid_m = sur_cfg.get("grid_m")
        grid_override = _build_surrogate_grid(fine_m, fine_h, grid_scale, grid_m)

    sur_solver = solver
    if eps_sur != float(solver.eps):
        sur_solver = EFGPSolver(
            kernel=solver.kernel,
            reg_lambda=solver.reg_lambda,
            eps=eps_sur,
            nufft_tol=solver.nufft_tol,
            l2scaled=solver.l2scaled,
        )

    backend_s, data_ctx_s, op_ctx_s, t_pre = _build_gpu_context(
        sur_solver,
        x_use,
        y_use,
        gpu_cfg,
        grid_override=grid_override,
    )

    mtot_sur = int(data_ctx_s.meta.get("mtot", 0))
    info = {
        "surrogate_tag": sur_tag,
        "surrogate_eps": float(eps_sur),
        "surrogate_m": int((mtot_sur - 1) // 2) if mtot_sur else np.nan,
        "surrogate_subsample_n": int(x_use.shape[0]),
        "surrogate_grid_override": bool(grid_override is not None),
        "surrogate_precompute": float(t_pre),
    }
    return sur_solver, backend_s, data_ctx_s, op_ctx_s, info


def _run_surrogate_eigenspace(sur_cfg, backend, data_ctx, op_ctx, top_q, solver, gpu_cfg):
    xp = backend.xp
    q_req = int(top_q) + 1

    sur_solver, backend_s, data_ctx_s, op_ctx_s, sur_info = _build_surrogate_context(
        solver,
        x_train,
        y_train,
        gpu_cfg,
        sur_cfg,
        data_ctx.meta,
    )

    def _apply_block_sur(V):
        return _apply_A_block_gpu(backend_s, data_ctx_s, op_ctx_s, REG_LAMBDA, V)

    sur_block = _resolve_block_size(q_req, sur_cfg)
    sur_iter = int(sur_cfg.get("sur_iter", 1))
    sur_iter = max(sur_iter, 1)
    eig_cfg_sur = EigenspaceConfig(q_max=q_req, block_size=sur_block, n_iter=sur_iter)

    t_s0 = time.perf_counter()
    vals_sur, vecs_sur, diag_sur = estimate_top_eigenspace_v3(
        backend=backend_s,
        apply_A_block_gpu=_apply_block_sur,
        size=int(data_ctx_s.rhs_gpu.size),
        cfg=eig_cfg_sur,
    )
    t_s1 = time.perf_counter()

    U_init = vecs_sur
    mtot_sur = int(data_ctx_s.meta.get("mtot", 0))
    mtot_fine = int(data_ctx.meta.get("mtot", 0))
    dim = int(data_ctx.meta.get("dim", 0))
    if mtot_sur and mtot_fine and mtot_sur != mtot_fine:
        U_init = _embed_coarse_to_fine(U_init, mtot_sur, mtot_fine, dim, xp)

    refine_iter = int(sur_cfg.get("refine_iter", 1))
    refine_iter = max(refine_iter, 1)
    fine_block = _resolve_block_size(q_req, sur_cfg)
    eig_cfg_fine = EigenspaceConfig(q_max=q_req, block_size=fine_block, n_iter=refine_iter)

    def _apply_block_fine(V):
        return _apply_A_block_gpu(backend, data_ctx, op_ctx, REG_LAMBDA, V)

    t_r0 = time.perf_counter()
    vals_fine, vecs_fine, diag_fine = estimate_top_eigenspace_v3(
        backend=backend,
        apply_A_block_gpu=_apply_block_fine,
        size=int(data_ctx.rhs_gpu.size),
        cfg=eig_cfg_fine,
        init_Q=U_init,
    )
    t_r1 = time.perf_counter()

    diag = dict(diag_fine)
    diag.update(sur_info)
    diag.update(
        {
            "surrogate_block_size": int(sur_block),
            "surrogate_n_iter": int(sur_iter),
            "surrogate_time_eigenspace": float(t_s1 - t_s0),
            "surrogate_refine_iter": int(refine_iter),
            "surrogate_time_refine": float(t_r1 - t_r0),
            "surrogate_diag_residual": float(diag_sur.get("residual_fro", np.nan)),
            "surrogate_diag_residual_rel": float(diag_sur.get("residual_fro_rel", np.nan)),
        }
    )
    return SimpleNamespace(values=vals_fine, vectors=vecs_fine), fine_block, diag


def _run_cupyx_eigenspace(cfg, backend, data_ctx, op_ctx, q_req):
    try:
        import cupyx.scipy.sparse.linalg as csl
    except Exception as exc:
        raise RuntimeError("cupyx.scipy.sparse.linalg is required for eigsh/lobpcg") from exc

    xp = backend.xp
    size = int(data_ctx.rhs_gpu.size)
    dtype = xp.complex128

    def _matvec(v):
        v = xp.asarray(v, dtype=dtype).reshape(-1)
        out = xp.empty_like(v)
        apply_A_v1(backend, data_ctx, v, float(REG_LAMBDA), op_ctx, out=out)
        return out

    def _matmat(X):
        X = xp.asarray(X, dtype=dtype)
        return _apply_A_block_gpu(backend, data_ctx, op_ctx, REG_LAMBDA, X)

    A = csl.LinearOperator(
        (size, size),
        matvec=_matvec,
        matmat=_matmat,
        dtype=dtype,
    )
    if cfg["method"] == "eigsh":
        eigvals, eigvecs = csl.eigsh(
            A,
            k=q_req,
            which="LM",
            tol=float(cfg.get("tol", 1e-6)),
            maxiter=cfg.get("maxiter", None),
        )
    else:
        X = _randn(xp, 0, (size, q_req)).astype(dtype)
        X, _ = xp.linalg.qr(X)
        X = xp.ascontiguousarray(X)
        eigvals, eigvecs = csl.lobpcg(
            A,
            X,
            tol=float(cfg.get("tol", 1e-6)),
            maxiter=cfg.get("maxiter", None),
            largest=True,
        )
    eigvals, eigvecs = _sort_eigpairs(eigvals, eigvecs, xp)
    return SimpleNamespace(values=eigvals, vectors=eigvecs)


def _run_eigenspace(cfg, backend, data_ctx, op_ctx, top_q, solver, gpu_cfg):
    xp = backend.xp
    q_req = int(top_q) + 1

    def _apply_block(V):
        return _apply_A_block_gpu(backend, data_ctx, op_ctx, REG_LAMBDA, V)

    def _apply_vec(v):
        out = _apply_A_block_gpu(backend, data_ctx, op_ctx, REG_LAMBDA, v)
        return out.reshape(-1)

    eig_diag = None
    if cfg["method"] == "v3_subspace_iter":
        block_size = _resolve_block_size(q_req, cfg)
        eig_cfg = EigenspaceConfig(
            q_max=q_req,
            block_size=block_size,
            n_iter=int(cfg.get("n_iter", 3)),
        )
        eigvals, eigvecs, eig_diag = estimate_top_eigenspace_v3(
            backend=backend,
            apply_A_block_gpu=_apply_block,
            size=int(data_ctx.rhs_gpu.size),
            cfg=eig_cfg,
        )
        eigpairs = SimpleNamespace(values=eigvals, vectors=eigvecs)
    elif cfg["method"] == "v3_eigenpro_nystrom":
        dummy_block = max(q_req + 1, 2)
        mcfg = {
            "data_ctx": data_ctx,
            "reg_lambda": float(REG_LAMBDA),
            "surrogate_lift": bool(cfg.get("surrogate_lift", True)),
        }
        _sbr = cfg.get("surrogate_block_rows", None)
        eig_cfg = EigenspaceConfig(
            q_max=q_req,
            block_size=dummy_block,
            n_iter=1,
            eig_method="eigenpro_nystrom",
            method_cfg=mcfg,
            surrogate_size=cfg.get("surrogate_size"),
            surrogate_oversample=int(cfg.get("surrogate_oversample", 10)),
            surrogate_lowfreq_ratio=float(cfg.get("surrogate_lowfreq_ratio", 0.25)),
            surrogate_seed=int(cfg.get("surrogate_seed", 0)),
            surrogate_block_rows=(int(_sbr) if _sbr is not None else None),
            surrogate_ritz_refine=bool(cfg.get("surrogate_ritz_refine", True)),
            surrogate_ritz_block_cols=int(cfg.get("surrogate_ritz_block_cols", 8)),
        )
        eigvals, eigvecs, eig_diag = estimate_top_eigenspace_v3(
            backend=backend,
            apply_A_block_gpu=_apply_block,
            size=int(data_ctx.rhs_gpu.size),
            cfg=eig_cfg,
        )
        block_size = int(eig_diag.get("block_size", dummy_block))
        eigpairs = SimpleNamespace(values=eigvals, vectors=eigvecs)
    elif cfg["method"] in ("eigsh", "lobpcg"):
        block_size = _resolve_block_size(q_req, cfg)
        eigpairs = _run_cupyx_eigenspace(cfg, backend, data_ctx, op_ctx, q_req)
    elif cfg["method"] == "surrogate":
        sur_cfg = cfg.get("surrogate", {})
        eigpairs, block_size, eig_diag = _run_surrogate_eigenspace(
            sur_cfg,
            backend,
            data_ctx,
            op_ctx,
            top_q,
            solver,
            gpu_cfg,
        )
    else:
        block_size = _resolve_block_size(q_req, cfg)
        fn = cfg.get("callable")
        if fn is None:
            raise ValueError(f"Unknown eigenspace method: {cfg.get('method')}")
        eigpairs = fn(
            matvec=_apply_vec,
            size=int(data_ctx.rhs_gpu.size),
            top_q=top_q,
            matvec_block=_apply_block,
            block_size=block_size,
            oversample=int(cfg.get("oversample", 2)),
            tol=float(cfg.get("tol", 1e-6)),
            maxiter=cfg.get("maxiter", None),
            xp=xp,
            cfg=cfg,
        )

    eigpairs = _normalize_eigpairs(eigpairs, xp)
    if eigpairs.values.size < q_req:
        raise ValueError("eigenspace must return at least top_q + 1 values")

    if eig_diag is None:
        res_fro, res_rel = _eigenspace_residual_gpu(
            _apply_block,
            eigpairs.values[:q_req],
            eigpairs.vectors[:, :q_req],
            xp,
        )
        eig_diag = {
            "residual_fro": res_fro,
            "residual_fro_rel": res_rel,
        }

    return eigpairs, block_size, eig_diag


def _eigenspace_residual_gpu(apply_block, eigvals, eigvecs, xp):
    AU = apply_block(eigvecs)
    resid = AU - eigvecs * eigvals.reshape(1, -1)
    res_fro = float(xp.linalg.norm(resid))
    au_norm = float(xp.linalg.norm(AU))
    res_rel = res_fro / max(au_norm, 1e-30)
    return res_fro, res_rel


def _build_precond_gpu(backend, eigvals, eigvecs, top_q, eig_diag=None):
    xp = backend.xp
    eigvals = xp.asarray(eigvals).reshape(-1)
    eigvecs = xp.asarray(eigvecs)
    q = int(top_q)
    ed = eig_diag if isinstance(eig_diag, dict) else {}
    mu = mu_for_precond_from_eig(eigvals, q, ed)
    scale = xp.asarray(1.0 - (mu / eigvals[:q]))
    precond = GPUPreconditionerData(
        U_gpu=eigvecs[:, :q],
        UH_gpu=eigvecs[:, :q].conj().T,
        scale_gpu=scale,
        scale_col_gpu=scale.reshape(-1, 1),
    )
    return precond, mu


def _run_pcg_gpu(backend, data_ctx, op_ctx, precond_data):
    def _matvec(v, out):
        apply_A_v1(backend, data_ctx, v, float(REG_LAMBDA), op_ctx, out=out)

    def _precond(v, out):
        apply_preconditioner_v2(backend, precond_data, v, op_ctx=op_ctx, out=out)

    return pcg_solve_gpu(
        backend,
        _matvec,
        _precond,
        data_ctx.rhs_gpu,
        op_ctx,
        GPU_TOL,
        GPU_MAXITER,
        return_stats=True,
    )


def _predict_gpu_host(backend, data_ctx, x_eval, beta_gpu):
    yhat_gpu = predict_v1(backend, data_ctx, x_eval, beta_gpu)
    return _device_to_numpy(backend.xp, yhat_gpu)


eigen esti

In [17]:
# Legacy custom eigenspace block kept for reference only.
# Active methods and the active registry are redefined in the next cell.

def custom_block_power(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    block_hint = int(block_size) if block_size is not None else int(top_q) + 1
    k, _, block, _ = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_hint,
        oversample=0,
        min_over=0,
    )
    n_iter = 8 if maxiter is None else int(maxiter)
    init_Q = _cfg_get(cfg, "init_Q", None)
    Q = _prepare_init_basis(xp, size, block, 0, xp.complex128, init_Q=init_Q)
    for _ in range(n_iter):
        Z = _apply_matvec_block(matvec, matvec_block, Q, xp)
        Q, _ = xp.linalg.qr(Z, mode="reduced")
    AQ = _apply_matvec_block(matvec, matvec_block, Q, xp)
    eigvals = xp.sum(xp.conj(Q) * AQ, axis=0)
    eigvals, eigvecs = _sort_eigpairs(eigvals, Q, xp)
    return SimpleNamespace(values=eigvals[:k], vectors=eigvecs[:, :k])


def custom_randomized_subspace(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, oversample, l, _ = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )
    n_iter = 3 if maxiter is None else int(maxiter)
    init_Q = _cfg_get(cfg, "init_Q", None)
    Q = _prepare_init_basis(xp, size, l, 1, xp.complex128, init_Q=init_Q)
    for _ in range(max(n_iter, 1)):
        Y = _apply_matvec_block(matvec, matvec_block, Q, xp)
        Q, _ = xp.linalg.qr(Y, mode="reduced")
    B = Q.conj().T @ _apply_matvec_block(matvec, matvec_block, Q, xp)
    B = 0.5 * (B + B.conj().T)
    eigvals, eigvecs_small = xp.linalg.eigh(B)
    eigvals = xp.real(eigvals)
    idx = xp.argsort(eigvals)[::-1]
    eigvals = eigvals[idx][:k]
    eigvecs = Q @ eigvecs_small[:, idx][:, :k]
    return SimpleNamespace(values=eigvals, vectors=eigvecs)


def _run_rr_on_basis(matvec_block, Q, k, xp, assume_orthonormal=False):
    Q = xp.asarray(Q, dtype=xp.complex128)
    if not assume_orthonormal:
        Q, _ = xp.linalg.qr(Q, mode="reduced")
    AQ = matvec_block(Q)
    B = Q.conj().T @ AQ
    B = 0.5 * (B + B.conj().T)
    eigvals, eigvecs_small = xp.linalg.eigh(B)
    eigvals = xp.real(eigvals)
    idx = xp.argsort(eigvals)[::-1]
    eigvals = eigvals[idx][:k]
    eigvecs = Q @ eigvecs_small[:, idx][:, :k]
    AU = AQ @ eigvecs_small[:, idx][:, :k]
    return SimpleNamespace(values=eigvals, vectors=eigvecs, Av=AU)


def _rr_residual_per_vec(matvec_block, eigvals, eigvecs, xp, AU=None):
    lam = xp.asarray(eigvals, dtype=xp.complex128).reshape(1, -1)
    U = xp.asarray(eigvecs, dtype=xp.complex128)
    if AU is None:
        AU = matvec_block(U)
    else:
        AU = xp.asarray(AU, dtype=xp.complex128)
    R = AU - U * lam
    nr = xp.linalg.norm(R, axis=0)
    na = xp.maximum(xp.linalg.norm(AU, axis=0), 1e-30)
    return xp.asarray(nr / na, dtype=xp.float64)


def _residual_converged(res_vec, k, tol_val, xp, q=0.90, max_factor=5.0):
    kk = min(int(k), int(res_vec.shape[0]))
    if kk <= 0:
        return False
    head = xp.asarray(res_vec[:kk], dtype=xp.float64)
    mx = float(xp.max(head))
    if not np.isfinite(mx):
        return False
    hs = xp.sort(head)
    qi = int(max(0, min(kk - 1, round(float(q) * (kk - 1)))))
    qv = float(hs[qi])
    return (qv <= float(tol_val)) and (mx <= float(max_factor) * float(tol_val))


def _cfg_get(cfg, key, default):
    if cfg is None:
        return default
    if isinstance(cfg, dict):
        return cfg.get(key, default)
    return getattr(cfg, key, default)


def _sanitize_subspace_dims(size, top_q, block_size, oversample, min_over=16):
    n = int(size)
    k = int(top_q) + 1
    if k < 1:
        raise ValueError("top_q must be >= 0.")
    if k >= n:
        raise ValueError(f"top_q is too large: k={k}, size={n}")

    over = max(int(oversample or 0), int(min_over))
    if block_size is None:
        block = k + over
    else:
        block = int(block_size)

    block = max(block, k)
    block = min(block, n)
    rr_rank = min(k + over, block)
    return k, over, block, rr_rank


def _sanitize_cutoff_bounds(lam_cut, lam_max, eps=1e-8):
    lam_cut = float(lam_cut)
    lam_max = float(lam_max)
    if not np.isfinite(lam_max) or lam_max <= 0.0:
        lam_max = 1.0
    if not np.isfinite(lam_cut) or lam_cut <= 0.0:
        lam_cut = 0.5 * lam_max
    if lam_cut >= lam_max:
        lam_cut = max(float(eps), 0.8 * lam_max)
    return lam_cut, lam_max


def _estimate_cutoff_bounds(matvec_block, size, k, block, xp, pilot_iters=2, pilot_block=None, rr_rank=None):
    if pilot_block is None:
        probe_cols = max(int(block), int(k) + 8)
    else:
        pilot_cols = max(1, min(int(block), int(pilot_block)))
        probe_cols = max(1, min(int(block), max(min(int(k) + 8, pilot_cols), min(8, pilot_cols))))
    rr_rank_eff = min(int(rr_rank) if rr_rank is not None else int(k), int(probe_cols))
    Qp = _randn(xp, 21, (size, probe_cols)).astype(xp.complex128)
    Qp, _ = xp.linalg.qr(Qp, mode="reduced")
    for _ in range(max(int(pilot_iters), 1)):
        Yp = matvec_block(Qp)
        Qp, _ = xp.linalg.qr(Yp, mode="reduced")
    rr = _run_rr_on_basis(matvec_block, Qp, min(probe_cols, size - 1), xp, assume_orthonormal=True)
    return _update_cutoff_from_ritz(rr.values, k, rr_rank_eff, xp)


def _update_cutoff_from_ritz(ritz_vals, k, rr_rank, xp):
    vals = xp.asarray(ritz_vals, dtype=xp.float64).reshape(-1)
    if vals.size == 0:
        return 0.0, 0.5, 1.0
    vals = xp.sort(vals)[::-1]
    lam_max = float(vals[0])
    lam_min = float(vals[-1])
    i0 = min(max(int(k) - 1, 0), vals.size - 1)
    i1 = min(max(int(rr_rank) - 1, i0), vals.size - 1)
    lam_cut = 0.5 * float(vals[i0] + vals[i1])
    lam_cut, lam_max = _sanitize_cutoff_bounds(lam_cut, lam_max)
    if lam_max <= lam_min:
        lam_min = min(lam_min, lam_max - 1e-6)
    return lam_min, lam_cut, lam_max


def _poly_filter_apply(matvec_block, V, degree, lam_cut, lam_max, xp):
    d = max(int(degree), 1)
    lam_cut, lam_max = _sanitize_cutoff_bounds(lam_cut, lam_max)
    scale = max(float(lam_max - lam_cut), 1e-12)
    inv_scale = float(1.0 / scale)
    shift_scale = float(-lam_cut / scale)

    def _Ahat(X, out=None, scratch=None):
        Xc = xp.asarray(X)
        Y = xp.asarray(matvec_block(Xc))
        if out is None:
            out = Y
        else:
            xp.copyto(out, Y)
        out *= inv_scale
        if scratch is None:
            scratch = xp.empty_like(out)
        xp.multiply(Xc, shift_scale, out=scratch)
        out += scratch
        return out, scratch

    T0 = xp.asarray(V)
    T1, scratch = _Ahat(T0)
    if d == 1:
        return T1

    work = xp.empty_like(T1)
    for _ in range(2, d + 1):
        work, scratch = _Ahat(T1, out=work, scratch=scratch)
        work *= 2.0
        work -= T0
        T0, T1, work = T1, work, T0
    return T1


def custom_mixed_precision_deflation(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    if xp is np:
        raise RuntimeError("custom_mixed_precision_deflation expects GPU backend.xp (cupy).")

    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, oversample, block, _ = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )
    n_iter = 20 if maxiter is None else int(maxiter)

    if matvec_block is None:
        def matvec_block(V):
            return _apply_matvec_block(matvec, None, V, xp)

    lowp_dtype = xp.complex64
    matvec_dtype = xp.complex64
    if str(_cfg_get(cfg, "lowp_dtype", "complex64")).lower() == "complex128":
        lowp_dtype = xp.complex128
    if str(_cfg_get(cfg, "matvec_dtype", "complex64")).lower() == "complex128":
        matvec_dtype = xp.complex128

    init_Q = _cfg_get(cfg, "init_Q", None)
    Q = _prepare_init_basis(xp, size, block, 11, lowp_dtype, init_Q=init_Q)
    tol_val = float(tol)
    rr_every = max(int(_cfg_get(cfg, "rr_check_every", 6)), 1)
    residual_every = int(_cfg_get(cfg, "residual_check_every", 0))
    rr_warmup_iters = max(int(_cfg_get(cfg, "rr_warmup_iters", max(rr_every, 8))), 1)
    reorth_every = max(int(_cfg_get(cfg, "reorth_every", 4)), 1)
    q_res = float(_cfg_get(cfg, "residual_q", 0.90))
    max_factor = float(_cfg_get(cfg, "residual_max_factor", 5.0))

    for it in range(max(n_iter, 1)):
        step = it + 1
        Qm = Q if getattr(Q, "dtype", None) == matvec_dtype else Q.astype(matvec_dtype, copy=False)
        try:
            Ym = matvec_block(Qm)
        except (TypeError, ValueError):
            Qm128 = Qm if getattr(Qm, "dtype", None) == xp.complex128 else Qm.astype(xp.complex128, copy=False)
            Ym = matvec_block(Qm128)
        should_rr = ((step >= rr_warmup_iters) and (step % rr_every == 0)) or (step == n_iter)
        use_highp_qr = should_rr or (step % reorth_every == 0)
        Q = _orthonormalize_iterate_mixed(Ym, xp, lowp_dtype, use_highp_qr)

        if should_rr:
            Qrr = Q if getattr(Q, "dtype", None) == xp.complex128 else Q.astype(xp.complex128, copy=False)
            rr = _run_rr_on_basis(matvec_block, Qrr, k, xp, assume_orthonormal=True)
            should_residual = (step == n_iter) or (residual_every > 0 and step >= rr_warmup_iters and step % residual_every == 0)
            if should_residual:
                res = _rr_residual_per_vec(matvec_block, rr.values, rr.vectors, xp, AU=getattr(rr, "Av", None))
                if _residual_converged(res, k, tol_val, xp, q=q_res, max_factor=max_factor):
                    return rr

    Qrr = Q if getattr(Q, "dtype", None) == xp.complex128 else Q.astype(xp.complex128, copy=False)
    return _run_rr_on_basis(matvec_block, Qrr, k, xp, assume_orthonormal=True)


def custom_poly_filter_svd(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    if xp is np:
        raise RuntimeError("custom_poly_filter_svd expects GPU backend.xp (cupy).")

    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, oversample, block, rr_rank = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )
    n_iter = 12 if maxiter is None else int(maxiter)
    poly_degree = int(_cfg_get(cfg, "poly_degree", 6))

    if matvec_block is None:
        def matvec_block(V):
            return _apply_matvec_block(matvec, None, V, xp)

    pilot_iters = int(_cfg_get(cfg, "pilot_iters", 2))
    pilot_block = int(_cfg_get(cfg, "pilot_block", min(block, 64)))
    _, lam_cut, lam_max = _estimate_cutoff_bounds(
        matvec_block,
        size,
        k,
        block,
        xp,
        pilot_iters=pilot_iters,
        pilot_block=pilot_block,
        rr_rank=rr_rank,
    )
    lam_cut, lam_max = _sanitize_cutoff_bounds(lam_cut, lam_max)
    cutoff_update_every = max(int(_cfg_get(cfg, "cutoff_update_every", 2)), 1)

    init_Q = _cfg_get(cfg, "init_Q", None)
    Q = _prepare_init_basis(xp, size, block, 12, xp.complex128, init_Q=init_Q)
    tol_val = float(tol)
    rr_every = max(int(_cfg_get(cfg, "rr_check_every", 6)), 1)
    residual_every = int(_cfg_get(cfg, "residual_check_every", 0))
    rr_warmup_iters = max(int(_cfg_get(cfg, "rr_warmup_iters", max(rr_every, 8))), 1)
    q_res = float(_cfg_get(cfg, "residual_q", 0.90))
    max_factor = float(_cfg_get(cfg, "residual_max_factor", 5.0))
    cutoff_warmup_iters = max(int(_cfg_get(cfg, "cutoff_warmup_iters", cutoff_update_every)), 1)
    for it in range(max(n_iter, 1)):
        Y = _poly_filter_apply(matvec_block, Q, poly_degree, lam_cut, lam_max, xp)
        Q, _ = xp.linalg.qr(Y, mode="reduced")
        step = it + 1
        should_update_cutoff = (
            (step >= cutoff_warmup_iters)
            and (step % cutoff_update_every == 0)
            and (step < n_iter)
        )
        if should_update_cutoff:
            rr_cut = _run_rr_on_basis(matvec_block, Q, rr_rank, xp, assume_orthonormal=True)
            _, lam_cut, lam_max = _update_cutoff_from_ritz(rr_cut.values, k, rr_rank, xp)
            lam_cut, lam_max = _sanitize_cutoff_bounds(lam_cut, lam_max)
        should_rr = ((step >= rr_warmup_iters) and (step % rr_every == 0)) or (step == n_iter)
        if should_rr:
            rr = _run_rr_on_basis(matvec_block, Q, k, xp, assume_orthonormal=True)
            should_residual = (step == n_iter) or (residual_every > 0 and step >= rr_warmup_iters and step % residual_every == 0)
            if should_residual:
                res = _rr_residual_per_vec(matvec_block, rr.values, rr.vectors, xp, AU=getattr(rr, "Av", None))
                if _residual_converged(res, k, tol_val, xp, q=q_res, max_factor=max_factor):
                    return rr

    return _run_rr_on_basis(matvec_block, Q, k, xp, assume_orthonormal=True)


def custom_mixed_poly_filter_svd(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    if xp is np:
        raise RuntimeError("custom_mixed_poly_filter_svd expects GPU backend.xp (cupy).")

    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, oversample, block, rr_rank = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )
    n_iter = 12 if maxiter is None else int(maxiter)
    poly_degree = int(_cfg_get(cfg, "poly_degree", 6))

    if matvec_block is None:
        def matvec_block(V):
            return _apply_matvec_block(matvec, None, V, xp)

    lowp_dtype = xp.complex64
    matvec_dtype = xp.complex64
    if str(_cfg_get(cfg, "lowp_dtype", "complex64")).lower() == "complex128":
        lowp_dtype = xp.complex128
    if str(_cfg_get(cfg, "matvec_dtype", "complex64")).lower() == "complex128":
        matvec_dtype = xp.complex128

    pilot_iters = int(_cfg_get(cfg, "pilot_iters", 2))
    pilot_block = int(_cfg_get(cfg, "pilot_block", min(block, 64)))
    _, lam_cut, lam_max = _estimate_cutoff_bounds(
        matvec_block,
        size,
        k,
        block,
        xp,
        pilot_iters=pilot_iters,
        pilot_block=pilot_block,
        rr_rank=rr_rank,
    )
    lam_cut, lam_max = _sanitize_cutoff_bounds(lam_cut, lam_max)
    cutoff_update_every = max(int(_cfg_get(cfg, "cutoff_update_every", 2)), 1)

    init_Q = _cfg_get(cfg, "init_Q", None)
    Q = _prepare_init_basis(xp, size, block, 13, lowp_dtype, init_Q=init_Q)
    tol_val = float(tol)
    rr_every = max(int(_cfg_get(cfg, "rr_check_every", 6)), 1)
    residual_every = int(_cfg_get(cfg, "residual_check_every", 0))
    rr_warmup_iters = max(int(_cfg_get(cfg, "rr_warmup_iters", max(rr_every, 8))), 1)
    reorth_every = max(int(_cfg_get(cfg, "reorth_every", 4)), 1)
    q_res = float(_cfg_get(cfg, "residual_q", 0.90))
    max_factor = float(_cfg_get(cfg, "residual_max_factor", 5.0))
    cutoff_warmup_iters = max(int(_cfg_get(cfg, "cutoff_warmup_iters", cutoff_update_every)), 1)
    for it in range(max(n_iter, 1)):
        step = it + 1
        Qm = Q if getattr(Q, "dtype", None) == matvec_dtype else Q.astype(matvec_dtype, copy=False)
        try:
            Ym = _poly_filter_apply(
                matvec_block,
                Qm,
                poly_degree,
                lam_cut,
                lam_max,
                xp,
            )
        except (TypeError, ValueError):
            Qm128 = Qm if getattr(Qm, "dtype", None) == xp.complex128 else Qm.astype(xp.complex128, copy=False)
            Ym = _poly_filter_apply(
                matvec_block,
                Qm128,
                poly_degree,
                lam_cut,
                lam_max,
                xp,
            )
        should_rr = ((step >= rr_warmup_iters) and (step % rr_every == 0)) or (step == n_iter)
        use_highp_qr = should_rr or (step % reorth_every == 0)
        Q = _orthonormalize_iterate_mixed(Ym, xp, lowp_dtype, use_highp_qr)
        should_update_cutoff = (
            (step >= cutoff_warmup_iters)
            and (step % cutoff_update_every == 0)
            and (step < n_iter)
        )
        if should_update_cutoff:
            Qrr = Q if getattr(Q, "dtype", None) == xp.complex128 else Q.astype(xp.complex128, copy=False)
            rr_cut = _run_rr_on_basis(matvec_block, Qrr, rr_rank, xp, assume_orthonormal=True)
            _, lam_cut, lam_max = _update_cutoff_from_ritz(rr_cut.values, k, rr_rank, xp)
            lam_cut, lam_max = _sanitize_cutoff_bounds(lam_cut, lam_max)
        if should_rr:
            Qrr = Q if getattr(Q, "dtype", None) == xp.complex128 else Q.astype(xp.complex128, copy=False)
            rr = _run_rr_on_basis(matvec_block, Qrr, k, xp, assume_orthonormal=True)
            should_residual = (step == n_iter) or (residual_every > 0 and step >= rr_warmup_iters and step % residual_every == 0)
            if should_residual:
                res = _rr_residual_per_vec(matvec_block, rr.values, rr.vectors, xp, AU=getattr(rr, "Av", None))
                if _residual_converged(res, k, tol_val, xp, q=q_res, max_factor=max_factor):
                    return rr

    Qrr = Q if getattr(Q, "dtype", None) == xp.complex128 else Q.astype(xp.complex128, copy=False)
    return _run_rr_on_basis(matvec_block, Qrr, k, xp, assume_orthonormal=True)


def _orthonormalize_block(Q, basis, xp):
    for B in basis:
        Q = Q - B @ (B.conj().T @ Q)
    Q, _ = xp.linalg.qr(Q, mode="reduced")
    return Q


def _pad_block(Q, block, xp):
    if Q.shape[1] >= block:
        return Q[:, :block]
    extra = _randn(xp, 2, (Q.shape[0], block - Q.shape[1])).astype(xp.complex128)
    Q = xp.concatenate([Q, extra], axis=1)
    Q, _ = xp.linalg.qr(Q, mode="reduced")
    return Q


def _build_gpu_linear_operator_from_callbacks(matvec, matvec_block, size, xp, dtype=None):
    try:
        import cupyx.scipy.sparse.linalg as csl
    except Exception as exc:
        raise RuntimeError("cupyx.scipy.sparse.linalg is required") from exc

    op_dtype = xp.complex128 if dtype is None else dtype

    def _matvec(v):
        vv = xp.asarray(v, dtype=op_dtype).reshape(-1)
        out = matvec(vv)
        if out is None:
            raise TypeError("matvec returned None")
        return xp.asarray(out, dtype=op_dtype).reshape(-1)

    def _matmat(V):
        VV = xp.asarray(V, dtype=op_dtype)
        out = _apply_matvec_block(matvec, matvec_block, VV, xp)
        if out is None:
            raise TypeError("matmat returned None")
        return xp.asarray(out, dtype=op_dtype)

    return csl.LinearOperator((size, size), matvec=_matvec, matmat=_matmat, dtype=op_dtype)


def _run_cupyx_eigsh_from_callbacks(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
    warm_start=False,
):
    xp = np if xp is None else xp
    if xp is np:
        raise RuntimeError("cupyx eigsh wrapper expects GPU backend.xp (cupy).")
    try:
        import cupyx.scipy.sparse.linalg as csl
    except Exception as exc:
        raise RuntimeError("cupyx.scipy.sparse.linalg is required for eigsh") from exc

    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, _, ncv_hint, _ = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )
    A = _build_gpu_linear_operator_from_callbacks(matvec, matvec_block, size, xp, dtype=xp.complex128)
    ncv = int(_cfg_get(cfg, "ncv", ncv_hint))
    ncv = max(int(k) + 2, ncv)
    ncv = min(int(size) - 1, ncv)
    init_Q = _cfg_get(cfg, "init_Q", None)
    v0 = None
    if init_Q is not None:
        V0 = xp.asarray(init_Q, dtype=xp.complex128)
        if V0.ndim == 2:
            v0 = xp.asarray(V0[:, 0], dtype=xp.complex128).reshape(-1)
        else:
            v0 = xp.asarray(V0, dtype=xp.complex128).reshape(-1)
    elif warm_start:
        rand_iters = int(_cfg_get(cfg, "rand_iters", 1))
        v0 = xp.asarray(_randn(xp, 41, (size,)), dtype=xp.complex128).reshape(-1)
        for _ in range(max(rand_iters, 1)):
            v0 = xp.asarray(matvec(v0), dtype=xp.complex128).reshape(-1)
            nv = xp.linalg.norm(v0)
            if float(nv) > 0.0:
                v0 = v0 / nv
    eigvals, eigvecs = csl.eigsh(
        A,
        k=int(k),
        which=str(_cfg_get(cfg, "which", "LA")),
        v0=v0,
        ncv=ncv,
        maxiter=maxiter,
        tol=float(tol),
        return_eigenvectors=True,
    )
    eigvals, eigvecs = _sort_eigpairs(eigvals, eigvecs, xp)
    return SimpleNamespace(values=eigvals[:k], vectors=eigvecs[:, :k])


def _block_krylov(matvec_block, Q0, n_steps, xp):
    basis = []
    Q = Q0
    for _ in range(max(int(n_steps), 1)):
        Q = _orthonormalize_block(Q, basis, xp)
        basis.append(Q)
        Q = matvec_block(Q)
    V = xp.concatenate(basis, axis=1)
    AV = matvec_block(V)
    T = V.conj().T @ AV
    T = 0.5 * (T + T.conj().T)
    eigvals, eigvecs = xp.linalg.eigh(T)
    idx = xp.argsort(eigvals)[::-1]
    eigvals = xp.real(eigvals[idx])
    eigvecs = eigvecs[:, idx]
    U = V @ eigvecs
    return eigvals, U


def _block_lanczos_core(matvec_block, Q0, k, block, krylov_depth, restart_cycles, xp):
    Q = _pad_block(Q0, block, xp)
    eigvals = None
    U = None
    for _ in range(max(int(restart_cycles), 1)):
        eigvals, U = _block_krylov(matvec_block, Q, krylov_depth, xp)
        U = U[:, :k]
        Q = _pad_block(U, block, xp)
    return eigvals[:k], U[:, :k]


def custom_block_lanczos(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    return _run_cupyx_eigsh_from_callbacks(
        matvec=matvec,
        size=size,
        top_q=top_q,
        matvec_block=matvec_block,
        block_size=block_size,
        oversample=oversample,
        tol=tol,
        maxiter=maxiter,
        xp=xp,
        cfg=cfg,
        warm_start=False,
    )


def custom_rand_lanczos(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    return _run_cupyx_eigsh_from_callbacks(
        matvec=matvec,
        size=size,
        top_q=top_q,
        matvec_block=matvec_block,
        block_size=block_size,
        oversample=oversample,
        tol=tol,
        maxiter=maxiter,
        xp=xp,
        cfg=cfg,
        warm_start=True,
    )


def custom_lobpcg_precond(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    if xp is np:
        raise RuntimeError("custom_lobpcg_precond expects GPU backend.xp (cupy).")
    try:
        import cupyx.scipy.sparse.linalg as csl
    except Exception as exc:
        raise RuntimeError("cupyx.scipy.sparse.linalg is required for lobpcg") from exc

    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, oversample, block, _ = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )

    dtype = xp.complex128

    def _matvec(v):
        v = xp.asarray(v, dtype=dtype).reshape(-1)
        out = matvec(v)
        if out is None:
            raise TypeError("matvec returned None in custom_lobpcg_precond")
        return xp.asarray(out, dtype=dtype).reshape(-1)

    def _matmat(V):
        V = xp.asarray(V, dtype=dtype)
        out = _apply_matvec_block(matvec, matvec_block, V, xp)
        if out is None:
            raise TypeError("matmat returned None in custom_lobpcg_precond")
        return xp.asarray(out, dtype=dtype)

    A = csl.LinearOperator((size, size), matvec=_matvec, matmat=_matmat, dtype=dtype)

    init_power_iters = int(_cfg_get(cfg, "init_power_iters", 2))
    init_Q = _cfg_get(cfg, "init_Q", None)
    X = _prepare_init_basis(xp, size, block, 5, dtype, init_Q=init_Q)
    for _ in range(max(init_power_iters, 1)):
        X = _matmat(X)
        X, _ = xp.linalg.qr(X, mode="reduced")

    precond = _cfg_get(cfg, "precond", None)
    precond_block = _cfg_get(cfg, "precond_block", None)
    M = None
    if precond is not None:
        def _pvec(v):
            v = xp.asarray(v, dtype=dtype).reshape(-1)
            out = precond(v)
            if out is None:
                raise TypeError("preconditioner returned None in custom_lobpcg_precond")
            return xp.asarray(out, dtype=dtype).reshape(-1)

        def _pmatmat(V):
            VV = xp.asarray(V, dtype=dtype)
            if precond_block is not None:
                out = precond_block(VV)
            else:
                cols = [_pvec(VV[:, i]) for i in range(int(VV.shape[1]))]
                out = xp.stack(cols, axis=1)
            if out is None:
                raise TypeError("preconditioner matmat returned None in custom_lobpcg_precond")
            return xp.asarray(out, dtype=dtype)
        M = csl.LinearOperator((size, size), matvec=_pvec, matmat=_pmatmat, dtype=dtype)

    try:
        eigvals, eigvecs = csl.lobpcg(
            A,
            X,
            M=M,
            tol=float(tol),
            maxiter=maxiter,
            largest=True,
        )
        eigvals, eigvecs = _sort_eigpairs(eigvals, eigvecs, xp)
        return SimpleNamespace(values=eigvals[:k], vectors=eigvecs[:, :k])
    except Exception as exc:
        print(f"[LOBPCG retry->fallback] {type(exc).__name__}: {exc}")
        fb_iter = int(_cfg_get(cfg, "fallback_maxiter", 3))
        return custom_randomized_subspace(
            matvec=matvec,
            size=size,
            top_q=top_q,
            matvec_block=matvec_block,
            block_size=block,
            oversample=oversample,
            tol=tol,
            maxiter=fb_iter,
            xp=xp,
            cfg=cfg,
        )


# Legacy registry disabled. Active custom method registry is defined in the next cell.
CUSTOM_EIGENSPACE_METHODS_LEGACY = []


def _register_custom_methods_legacy_disabled():
    return None


In [18]:
# Refactored custom eigenspace methods


def _make_highp_block_matvec(matvec, matvec_block, xp, cfg=None):
    explicit = _cfg_get(cfg, "matvec_block_highp", None)
    if explicit is not None:
        def _highp(V):
            out = explicit(xp.asarray(V, dtype=xp.complex128))
            if out is None:
                raise TypeError("matvec_block_highp returned None")
            return xp.asarray(out, dtype=xp.complex128)
        return _highp

    def _default(V):
        Vh = xp.asarray(V, dtype=xp.complex128)
        out = _apply_matvec_block(matvec, matvec_block, Vh, xp)
        if out is None:
            raise TypeError("matvec_block returned None")
        return xp.asarray(out, dtype=xp.complex128)

    return _default


def _make_lowp_block_matvec(xp, cfg=None, lowp_dtype=None, matvec_block=None):
    explicit = _cfg_get(cfg, "matvec_block_lowp", None)
    if explicit is not None:
        def _lowp_explicit(V):
            out = explicit(xp.asarray(V, dtype=lowp_dtype))
            if out is None:
                raise TypeError("matvec_block_lowp returned None")
            return xp.asarray(out, dtype=lowp_dtype)
        return _lowp_explicit

    allow_implicit = bool(_cfg_get(cfg, "allow_implicit_lowp", False))
    if allow_implicit and matvec_block is not None:
        def _lowp_implicit(V):
            Vl = xp.asarray(V, dtype=lowp_dtype)
            out = matvec_block(Vl)
            if out is None:
                raise TypeError("implicit lowp matvec_block returned None")
            return xp.asarray(out, dtype=lowp_dtype)
        return _lowp_implicit

    return None


def _run_rr_check(
    matvec_block_highp,
    Q,
    k,
    xp,
    step,
    n_iter,
    tol_val,
    rr_every,
    residual_every,
    rr_warmup_iters,
    q_res,
    max_factor,
):
    should_rr = ((step >= rr_warmup_iters) and (step % rr_every == 0)) or (step == n_iter)
    if not should_rr:
        return None, False

    Qrr = xp.asarray(Q, dtype=xp.complex128)
    rr = _run_rr_on_basis(matvec_block_highp, Qrr, k, xp, assume_orthonormal=True)
    should_residual = (step == n_iter) or (
        residual_every > 0 and step >= rr_warmup_iters and step % residual_every == 0
    )
    converged = False
    if should_residual:
        res = _rr_residual_per_vec(
            matvec_block_highp,
            rr.values,
            rr.vectors,
            xp,
            AU=getattr(rr, "Av", None),
        )
        rr.residual = res
        converged = _residual_converged(
            res,
            k,
            tol_val,
            xp,
            q=q_res,
            max_factor=max_factor,
        )
    return rr, converged


def _attach_solver_stats(result, **stats):
    if result is None:
        return None
    for key, value in stats.items():
        setattr(result, key, value)
    return result


def block_power_rr(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    block_hint = int(block_size) if block_size is not None else int(top_q) + 1
    k, _, block, _ = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_hint,
        oversample=0,
        min_over=0,
    )
    n_iter = int(_cfg_get(cfg, "maxiter", 8 if maxiter is None else maxiter))
    tol_val = float(_cfg_get(cfg, "tol", tol))
    rr_every = max(int(_cfg_get(cfg, "rr_check_every", 1)), 1)
    residual_every = max(int(_cfg_get(cfg, "residual_check_every", 1)), 1)
    rr_warmup_iters = max(int(_cfg_get(cfg, "rr_warmup_iters", 1)), 1)
    q_res = float(_cfg_get(cfg, "residual_q", 0.90))
    max_factor = float(_cfg_get(cfg, "residual_max_factor", 5.0))
    init_Q = _cfg_get(cfg, "init_Q", None)
    matvec_block_highp = _make_highp_block_matvec(matvec, matvec_block, xp, cfg)

    Q = _prepare_init_basis(xp, size, block, 0, xp.complex128, init_Q=init_Q)
    rr = None
    for it in range(max(n_iter, 1)):
        step = it + 1
        Z = matvec_block_highp(Q)
        Q, _ = xp.linalg.qr(Z, mode="reduced")
        rr, converged = _run_rr_check(
            matvec_block_highp,
            Q,
            k,
            xp,
            step,
            n_iter,
            tol_val,
            rr_every,
            residual_every,
            rr_warmup_iters,
            q_res,
            max_factor,
        )
        if rr is not None and converged:
            return rr

    if rr is None:
        rr = _run_rr_on_basis(matvec_block_highp, Q, k, xp, assume_orthonormal=True)
    return rr


def rand_subspace_rr(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, _, block, _ = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )
    n_iter = int(_cfg_get(cfg, "maxiter", 3 if maxiter is None else maxiter))
    tol_val = float(_cfg_get(cfg, "tol", tol))
    rr_every = max(int(_cfg_get(cfg, "rr_check_every", 1)), 1)
    residual_every = max(int(_cfg_get(cfg, "residual_check_every", 1)), 1)
    rr_warmup_iters = max(int(_cfg_get(cfg, "rr_warmup_iters", 1)), 1)
    q_res = float(_cfg_get(cfg, "residual_q", 0.90))
    max_factor = float(_cfg_get(cfg, "residual_max_factor", 5.0))
    init_Q = _cfg_get(cfg, "init_Q", None)
    matvec_block_highp = _make_highp_block_matvec(matvec, matvec_block, xp, cfg)

    Q = _prepare_init_basis(xp, size, block, 1, xp.complex128, init_Q=init_Q)
    rr = None
    for it in range(max(n_iter, 1)):
        step = it + 1
        Y = matvec_block_highp(Q)
        Q, _ = xp.linalg.qr(Y, mode="reduced")
        rr, converged = _run_rr_check(
            matvec_block_highp,
            Q,
            k,
            xp,
            step,
            n_iter,
            tol_val,
            rr_every,
            residual_every,
            rr_warmup_iters,
            q_res,
            max_factor,
        )
        if rr is not None and converged:
            return rr

    if rr is None:
        rr = _run_rr_on_basis(matvec_block_highp, Q, k, xp, assume_orthonormal=True)
    return rr


def mixed_subspace_rr(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    if xp is np:
        raise RuntimeError("mixed_subspace_rr expects GPU backend.xp (cupy).")

    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, _, block, _ = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )
    n_iter = int(_cfg_get(cfg, "maxiter", 20 if maxiter is None else maxiter))
    tol_val = float(_cfg_get(cfg, "tol", tol))
    rr_every = max(int(_cfg_get(cfg, "rr_check_every", 4)), 1)
    residual_every = max(int(_cfg_get(cfg, "residual_check_every", 4)), 1)
    rr_warmup_iters = max(int(_cfg_get(cfg, "rr_warmup_iters", rr_every)), 1)
    reorth_every = max(int(_cfg_get(cfg, "reorth_every", 4)), 1)
    q_res = float(_cfg_get(cfg, "residual_q", 0.90))
    max_factor = float(_cfg_get(cfg, "residual_max_factor", 5.0))

    lowp_dtype = xp.complex64
    if str(_cfg_get(cfg, "lowp_dtype", "complex64")).lower() == "complex128":
        lowp_dtype = xp.complex128
    matvec_dtype = xp.complex64
    if str(_cfg_get(cfg, "matvec_dtype", "complex64")).lower() == "complex128":
        matvec_dtype = xp.complex128

    init_Q = _cfg_get(cfg, "init_Q", None)
    matvec_block_highp = _make_highp_block_matvec(matvec, matvec_block, xp, cfg)
    _raw_mb = matvec_block if matvec_block is not None else (lambda V: _apply_matvec_block(matvec, None, V, xp))
    matvec_block_lowp = _make_lowp_block_matvec(xp, cfg, matvec_dtype, matvec_block=_raw_mb)

    Q = _prepare_init_basis(xp, size, block, 11, lowp_dtype, init_Q=init_Q)
    rr = None
    lowp_steps = 0
    highp_steps = 0
    for it in range(max(n_iter, 1)):
        step = it + 1
        Qm = Q if getattr(Q, "dtype", None) == matvec_dtype else Q.astype(matvec_dtype, copy=False)
        if matvec_block_lowp is not None:
            Ym = matvec_block_lowp(Qm)
            lowp_steps += 1
        else:
            Ym = matvec_block_highp(Qm)
            highp_steps += 1
        should_rr = ((step >= rr_warmup_iters) and (step % rr_every == 0)) or (step == n_iter)
        use_highp_qr = should_rr or (step % reorth_every == 0)
        Q = _orthonormalize_iterate_mixed(Ym, xp, lowp_dtype, use_highp_qr)
        rr, converged = _run_rr_check(
            matvec_block_highp,
            Q,
            k,
            xp,
            step,
            n_iter,
            tol_val,
            rr_every,
            residual_every,
            rr_warmup_iters,
            q_res,
            max_factor,
        )
        if rr is not None and converged:
            return _attach_solver_stats(rr, lowp_steps=lowp_steps, highp_steps=highp_steps)

    if rr is None:
        Qrr = xp.asarray(Q, dtype=xp.complex128)
        rr = _run_rr_on_basis(matvec_block_highp, Qrr, k, xp, assume_orthonormal=True)
    return _attach_solver_stats(rr, lowp_steps=lowp_steps, highp_steps=highp_steps)


def _adaptive_stopband_min(lam_min_est, lam_cut, cfg):
    explicit = _cfg_get(cfg, "stopband_min", None)
    if explicit is not None:
        v = float(explicit)
        if np.isfinite(v):
            return max(v, 0.0)
    if np.isfinite(lam_min_est) and lam_min_est > 0.0:
        return max(0.0, float(lam_min_est) * 0.95)
    return 0.0


def _adaptive_poly_degree(stopband_min, lam_cut, lam_max, cfg):
    explicit = _cfg_get(cfg, "poly_degree", None)
    if explicit is not None:
        d = int(explicit)
        if d > 0:
            return d
    a = float(stopband_min)
    b = float(lam_cut)
    m = float(lam_max)
    stop_width = max(b - a, 1e-12)
    pass_width = max(m - b, 1e-12)
    ratio = pass_width / stop_width
    if ratio >= 1.0:
        return 4
    elif ratio >= 0.3:
        return 6
    elif ratio >= 0.1:
        return 8
    else:
        return 12


def _chebyshev_scalar(degree, x):
    degree = int(max(degree, 0))
    x = float(x)
    if degree == 0:
        return 1.0
    if x <= -1.0:
        return ((-1.0) ** degree) * np.cosh(degree * np.arccosh(abs(x)))
    if abs(x) <= 1.0:
        return float(np.cos(degree * np.arccos(x)))
    return float(np.cosh(degree * np.arccosh(x)))


def _poly_filter_apply_chebyshev(matvec_block, V, degree, stop_a, stop_b, lam_max, xp):
    d = int(max(degree, 0))
    a = float(stop_a)
    b = float(stop_b)
    lam_max = float(lam_max)
    if not np.isfinite(a):
        a = 0.0
    if not np.isfinite(b) or b <= a:
        b = max(a + 1e-8, 0.5 * max(lam_max, 1.0))
    if lam_max <= b:
        lam_max = b + max(1e-8, 1e-3 * max(abs(b), 1.0))

    c = 0.5 * (a + b)
    e = 0.5 * (b - a)
    if e <= 0.0:
        raise ValueError("invalid Chebyshev stopband")

    def _Ahat(X):
        Xh = xp.asarray(X)
        Y = xp.asarray(matvec_block(Xh))
        return (Y - c * Xh) / e

    T0 = xp.asarray(V)
    if d == 0:
        return T0

    T1 = _Ahat(T0)
    if d == 1:
        out = T1
    else:
        for _ in range(2, d + 1):
            T2 = 2.0 * _Ahat(T1) - T0
            T0, T1 = T1, T2
        out = T1

    sigma = max((lam_max - c) / e, 1.0 + 1e-12)
    rho = _chebyshev_scalar(d, sigma)
    if not np.isfinite(rho) or abs(rho) < 1e-12:
        rho = 1.0
    return out / float(rho)


def poly_filtered_subspace(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    if xp is np:
        raise RuntimeError("poly_filtered_subspace expects GPU backend.xp (cupy).")

    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, _, block, rr_rank = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )
    n_iter = int(_cfg_get(cfg, "maxiter", 10 if maxiter is None else maxiter))
    pilot_iters = int(_cfg_get(cfg, "pilot_iters", 1))
    pilot_block = int(_cfg_get(cfg, "pilot_block", min(block, 64)))
    tol_val = float(_cfg_get(cfg, "tol", tol))
    rr_every = max(int(_cfg_get(cfg, "rr_check_every", 2)), 1)
    residual_every = max(int(_cfg_get(cfg, "residual_check_every", 2)), 1)
    rr_warmup_iters = max(int(_cfg_get(cfg, "rr_warmup_iters", rr_every)), 1)
    cutoff_update_every = max(int(_cfg_get(cfg, "cutoff_update_every", 2)), 1)
    cutoff_warmup_iters = max(int(_cfg_get(cfg, "cutoff_warmup_iters", 2)), 1)
    q_res = float(_cfg_get(cfg, "residual_q", 0.90))
    max_factor = float(_cfg_get(cfg, "residual_max_factor", 5.0))
    init_Q = _cfg_get(cfg, "init_Q", None)

    matvec_block_highp = _make_highp_block_matvec(matvec, matvec_block, xp, cfg)
    lam_min_est, lam_cut, lam_max = _estimate_cutoff_bounds(
        matvec_block_highp,
        size,
        k,
        block,
        xp,
        pilot_iters=pilot_iters,
        pilot_block=pilot_block,
        rr_rank=rr_rank,
    )
    lam_cut, lam_max = _sanitize_cutoff_bounds(lam_cut, lam_max)
    stopband_min = _adaptive_stopband_min(lam_min_est, lam_cut, cfg)
    poly_degree = _adaptive_poly_degree(stopband_min, lam_cut, lam_max, cfg)

    Q = _prepare_init_basis(xp, size, block, 12, xp.complex128, init_Q=init_Q)
    rr = None
    for it in range(max(n_iter, 1)):
        step = it + 1
        Y = _poly_filter_apply_chebyshev(
            matvec_block_highp,
            Q,
            poly_degree,
            stopband_min,
            lam_cut,
            lam_max,
            xp,
        )
        Q, _ = xp.linalg.qr(Y, mode="reduced")

        should_update_cutoff = (
            step >= cutoff_warmup_iters
            and step % cutoff_update_every == 0
            and step < n_iter
        )
        if should_update_cutoff:
            rr_cut = _run_rr_on_basis(matvec_block_highp, Q, rr_rank, xp, assume_orthonormal=True)
            lam_min_est, lam_cut, lam_max = _update_cutoff_from_ritz(rr_cut.values, k, rr_rank, xp)
            lam_cut, lam_max = _sanitize_cutoff_bounds(lam_cut, lam_max)
            stopband_min = _adaptive_stopband_min(lam_min_est, lam_cut, cfg)
            poly_degree = _adaptive_poly_degree(stopband_min, lam_cut, lam_max, cfg)

        rr, converged = _run_rr_check(
            matvec_block_highp,
            Q,
            k,
            xp,
            step,
            n_iter,
            tol_val,
            rr_every,
            residual_every,
            rr_warmup_iters,
            q_res,
            max_factor,
        )
        if rr is not None and converged:
            return rr

    if rr is None:
        rr = _run_rr_on_basis(matvec_block_highp, Q, k, xp, assume_orthonormal=True)
    return rr


def mixed_poly_filtered_subspace(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    if xp is np:
        raise RuntimeError("mixed_poly_filtered_subspace expects GPU backend.xp (cupy).")

    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, _, block, rr_rank = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )
    n_iter = int(_cfg_get(cfg, "maxiter", 10 if maxiter is None else maxiter))
    pilot_iters = int(_cfg_get(cfg, "pilot_iters", 1))
    pilot_block = int(_cfg_get(cfg, "pilot_block", min(block, 64)))
    tol_val = float(_cfg_get(cfg, "tol", tol))
    rr_every = max(int(_cfg_get(cfg, "rr_check_every", 2)), 1)
    residual_every = max(int(_cfg_get(cfg, "residual_check_every", 2)), 1)
    rr_warmup_iters = max(int(_cfg_get(cfg, "rr_warmup_iters", rr_every)), 1)
    reorth_every = max(int(_cfg_get(cfg, "reorth_every", 4)), 1)
    cutoff_update_every = max(int(_cfg_get(cfg, "cutoff_update_every", 2)), 1)
    cutoff_warmup_iters = max(int(_cfg_get(cfg, "cutoff_warmup_iters", 2)), 1)
    q_res = float(_cfg_get(cfg, "residual_q", 0.90))
    max_factor = float(_cfg_get(cfg, "residual_max_factor", 5.0))

    lowp_dtype = xp.complex64
    if str(_cfg_get(cfg, "lowp_dtype", "complex64")).lower() == "complex128":
        lowp_dtype = xp.complex128
    matvec_dtype = xp.complex64
    if str(_cfg_get(cfg, "matvec_dtype", "complex64")).lower() == "complex128":
        matvec_dtype = xp.complex128

    init_Q = _cfg_get(cfg, "init_Q", None)
    matvec_block_highp = _make_highp_block_matvec(matvec, matvec_block, xp, cfg)
    _raw_mb_mp = matvec_block if matvec_block is not None else (lambda V: _apply_matvec_block(matvec, None, V, xp))
    matvec_block_lowp = _make_lowp_block_matvec(xp, cfg, matvec_dtype, matvec_block=_raw_mb_mp)
    lam_min_est, lam_cut, lam_max = _estimate_cutoff_bounds(
        matvec_block_highp,
        size,
        k,
        block,
        xp,
        pilot_iters=pilot_iters,
        pilot_block=pilot_block,
        rr_rank=rr_rank,
    )
    lam_cut, lam_max = _sanitize_cutoff_bounds(lam_cut, lam_max)
    stopband_min = _adaptive_stopband_min(lam_min_est, lam_cut, cfg)
    poly_degree = _adaptive_poly_degree(stopband_min, lam_cut, lam_max, cfg)

    Q = _prepare_init_basis(xp, size, block, 13, lowp_dtype, init_Q=init_Q)
    rr = None
    lowp_steps = 0
    highp_steps = 0
    for it in range(max(n_iter, 1)):
        step = it + 1
        Qm = Q if getattr(Q, "dtype", None) == matvec_dtype else Q.astype(matvec_dtype, copy=False)
        filter_matvec = matvec_block_lowp if matvec_block_lowp is not None else matvec_block_highp
        Ym = _poly_filter_apply_chebyshev(
            filter_matvec,
            Qm,
            poly_degree,
            stopband_min,
            lam_cut,
            lam_max,
            xp,
        )
        if matvec_block_lowp is not None:
            lowp_steps += 1
        else:
            highp_steps += 1

        should_rr = ((step >= rr_warmup_iters) and (step % rr_every == 0)) or (step == n_iter)
        use_highp_qr = should_rr or (step % reorth_every == 0)
        Q = _orthonormalize_iterate_mixed(Ym, xp, lowp_dtype, use_highp_qr)

        should_update_cutoff = (
            step >= cutoff_warmup_iters
            and step % cutoff_update_every == 0
            and step < n_iter
        )
        if should_update_cutoff:
            Qrr = xp.asarray(Q, dtype=xp.complex128)
            rr_cut = _run_rr_on_basis(matvec_block_highp, Qrr, rr_rank, xp, assume_orthonormal=True)
            lam_min_est, lam_cut, lam_max = _update_cutoff_from_ritz(rr_cut.values, k, rr_rank, xp)
            lam_cut, lam_max = _sanitize_cutoff_bounds(lam_cut, lam_max)
            stopband_min = _adaptive_stopband_min(lam_min_est, lam_cut, cfg)
            poly_degree = _adaptive_poly_degree(stopband_min, lam_cut, lam_max, cfg)

        rr, converged = _run_rr_check(
            matvec_block_highp,
            Q,
            k,
            xp,
            step,
            n_iter,
            tol_val,
            rr_every,
            residual_every,
            rr_warmup_iters,
            q_res,
            max_factor,
        )
        if rr is not None and converged:
            return _attach_solver_stats(rr, lowp_steps=lowp_steps, highp_steps=highp_steps)

    if rr is None:
        Qrr = xp.asarray(Q, dtype=xp.complex128)
        rr = _run_rr_on_basis(matvec_block_highp, Qrr, k, xp, assume_orthonormal=True)
    return _attach_solver_stats(rr, lowp_steps=lowp_steps, highp_steps=highp_steps)


def _build_eigsh_start_vector(matvec_block_highp, size, xp, cfg=None):
    strategy = str(_cfg_get(cfg, "warm_start_strategy", "none") or "none").lower()
    if strategy == "none":
        return None
    if strategy == "init_q_first_col":
        init_Q = _cfg_get(cfg, "init_Q", None)
        if init_Q is None:
            return None
        V0 = xp.asarray(init_Q, dtype=xp.complex128)
        if V0.ndim == 2:
            V0 = V0[:, 0]
        return xp.asarray(V0, dtype=xp.complex128).reshape(-1)

    v0 = xp.asarray(_randn(xp, 41, (size,)), dtype=xp.complex128).reshape(-1)
    nv = xp.linalg.norm(v0)
    if float(nv) > 0.0:
        v0 = v0 / nv
    if strategy == "random":
        return v0
    if strategy == "power1":
        V1 = matvec_block_highp(v0.reshape(-1, 1))[:, 0]
        nv = xp.linalg.norm(V1)
        if float(nv) > 0.0:
            V1 = V1 / nv
        return xp.asarray(V1, dtype=xp.complex128).reshape(-1)
    raise ValueError(f"unknown warm_start_strategy: {strategy}")


def cupy_eigsh(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    if xp is np:
        raise RuntimeError("cupy_eigsh expects GPU backend.xp (cupy).")
    try:
        import cupyx.scipy.sparse.linalg as csl
    except Exception as exc:
        raise RuntimeError("cupyx.scipy.sparse.linalg is required for eigsh") from exc

    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, _, ncv_hint, _ = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )
    matvec_block_highp = _make_highp_block_matvec(matvec, matvec_block, xp, cfg)
    op_dtype = xp.complex128

    def _eigsh_mv(v):
        vv = xp.asarray(v, dtype=op_dtype).reshape(-1, 1)
        out = matvec_block_highp(vv)
        return xp.asarray(out[:, 0], dtype=op_dtype).reshape(-1)

    def _eigsh_mm(V):
        VV = xp.asarray(V, dtype=op_dtype)
        return xp.asarray(matvec_block_highp(VV), dtype=op_dtype)

    A = csl.LinearOperator((size, size), matvec=_eigsh_mv, matmat=_eigsh_mm, dtype=op_dtype)
    ncv = _cfg_get(cfg, "ncv", None)
    if ncv is None:
        ncv = max(2 * int(k) + 32, int(k) + 2)
    ncv = max(int(k) + 2, int(ncv))
    ncv = min(int(size) - 1, int(ncv))
    maxiter_eff = _cfg_get(cfg, "maxiter", maxiter)
    v0 = _build_eigsh_start_vector(matvec_block_highp, size, xp, cfg)

    eigvals, eigvecs = csl.eigsh(
        A,
        k=int(k),
        which=str(_cfg_get(cfg, "which", "LA")),
        v0=v0,
        ncv=ncv,
        maxiter=maxiter_eff,
        tol=float(_cfg_get(cfg, "tol", tol)),
        return_eigenvectors=True,
    )
    eigvals, eigvecs = _sort_eigpairs(eigvals, eigvecs, xp)
    return SimpleNamespace(values=eigvals[:k], vectors=eigvecs[:, :k])


def _build_refine_init_basis(
    matvec,
    size,
    top_q,
    matvec_block,
    block_size,
    oversample,
    tol,
    xp,
    cfg,
):
    k, _, _, _ = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=0,
    )
    dtype = xp.complex128
    init_Q = _cfg_get(cfg, "init_Q", None)
    if init_Q is not None:
        return _prepare_init_basis(xp, size, k, 5, dtype, init_Q=init_Q)

    init_source = str(_cfg_get(cfg, "init_source", "rand_subspace_rr") or "rand_subspace_rr")
    init_maxiter = int(_cfg_get(cfg, "fallback_maxiter", 3))
    init_cfg = dict(cfg) if isinstance(cfg, dict) else {}
    init_cfg["init_Q"] = None
    init_cfg["maxiter"] = min(int(_cfg_get(cfg, "maxiter", init_maxiter) or init_maxiter), init_maxiter)

    if init_source == "rand_subspace_rr":
        init_res = rand_subspace_rr(matvec, size, top_q, matvec_block, block_size, oversample, tol, init_maxiter, xp, init_cfg)
        X = init_res.vectors[:, :k]
    elif init_source == "mixed_subspace_rr":
        init_res = mixed_subspace_rr(matvec, size, top_q, matvec_block, block_size, oversample, tol, init_maxiter, xp, init_cfg)
        X = init_res.vectors[:, :k]
    elif init_source == "poly_filtered_subspace":
        init_res = poly_filtered_subspace(matvec, size, top_q, matvec_block, block_size, oversample, tol, init_maxiter, xp, init_cfg)
        X = init_res.vectors[:, :k]
    elif init_source == "mixed_poly_filtered_subspace":
        init_res = mixed_poly_filtered_subspace(matvec, size, top_q, matvec_block, block_size, oversample, tol, init_maxiter, xp, init_cfg)
        X = init_res.vectors[:, :k]
    else:
        X = _prepare_init_basis(xp, size, k, 5, dtype, init_Q=None)

    X = xp.asarray(X, dtype=dtype)
    if int(X.shape[1]) != int(k):
        X = _prepare_init_basis(xp, size, k, 5, dtype, init_Q=X)
    X, _ = xp.linalg.qr(X, mode="reduced")
    return X[:, :k]


def cupy_lobpcg_refine(
    matvec,
    size,
    top_q,
    matvec_block=None,
    block_size=None,
    oversample=2,
    tol=1e-6,
    maxiter=None,
    xp=None,
    cfg=None,
):
    xp = np if xp is None else xp
    if xp is np:
        raise RuntimeError("cupy_lobpcg_refine expects GPU backend.xp (cupy).")
    try:
        import cupyx.scipy.sparse.linalg as csl
    except Exception as exc:
        raise RuntimeError("cupyx.scipy.sparse.linalg is required for lobpcg") from exc

    min_over = int(_cfg_get(cfg, "min_oversample", 16))
    k, _, _, _ = _sanitize_subspace_dims(
        size=size,
        top_q=top_q,
        block_size=block_size,
        oversample=oversample,
        min_over=min_over,
    )
    dtype = xp.complex128
    matvec_block_highp = _make_highp_block_matvec(matvec, matvec_block, xp, cfg)

    def _matvec(v):
        vv = xp.asarray(v, dtype=dtype).reshape(-1, 1)
        out = matvec_block_highp(vv)
        if out is None:
            raise TypeError("matvec_block_highp returned None in cupy_lobpcg_refine")
        return xp.asarray(out[:, 0], dtype=dtype).reshape(-1)

    def _matmat(V):
        VV = xp.asarray(V, dtype=dtype)
        out = matvec_block_highp(VV)
        if out is None:
            raise TypeError("matvec_block_highp returned None in cupy_lobpcg_refine (matmat)")
        return xp.asarray(out, dtype=dtype)

    A = csl.LinearOperator((size, size), matvec=_matvec, matmat=_matmat, dtype=dtype)
    X = _build_refine_init_basis(
        matvec,
        size,
        top_q,
        matvec_block,
        block_size,
        oversample,
        tol,
        xp,
        cfg,
    )
    init_power_iters = max(int(_cfg_get(cfg, "init_power_iters", 0)), 0)
    for _ in range(init_power_iters):
        X = xp.asarray(matvec_block_highp(X), dtype=dtype)
        X, _ = xp.linalg.qr(X, mode="reduced")
        X = X[:, :k]

    use_precond = bool(_cfg_get(cfg, "use_precond", True))
    allow_unpreconditioned = bool(_cfg_get(cfg, "allow_unpreconditioned_lobpcg", False))
    precond = _cfg_get(cfg, "precond", None)
    precond_block = _cfg_get(cfg, "precond_block", None)
    M = None
    if use_precond and precond is not None:
        def _pvec(v):
            vv = xp.asarray(v, dtype=dtype).reshape(-1)
            out = precond(vv)
            if out is None:
                raise TypeError("preconditioner returned None in cupy_lobpcg_refine")
            return xp.asarray(out, dtype=dtype).reshape(-1)

        def _pmatmat(V):
            VV = xp.asarray(V, dtype=dtype)
            if precond_block is not None:
                out = precond_block(VV)
            else:
                cols = [_pvec(VV[:, i]) for i in range(int(VV.shape[1]))]
                out = xp.stack(cols, axis=1)
            if out is None:
                raise TypeError("preconditioner matmat returned None in cupy_lobpcg_refine")
            return xp.asarray(out, dtype=dtype)

        M = csl.LinearOperator((size, size), matvec=_pvec, matmat=_pmatmat, dtype=dtype)

    # Ensure X is a clean contiguous complex128 array (CuPy lobpcg is sensitive to views)
    X_lobpcg = xp.ascontiguousarray(xp.asarray(X[:, :k], dtype=dtype))
    _maxiter_raw = _cfg_get(cfg, "maxiter", maxiter)
    _maxiter_lobpcg = int(_maxiter_raw) if (_maxiter_raw is not None) else 40
    _tol_lobpcg = float(_cfg_get(cfg, "tol", tol) or tol)

    try:
        lobpcg_kwargs = {
            "tol": _tol_lobpcg,
            "maxiter": _maxiter_lobpcg,
            "largest": True,
        }
        if M is not None:
            lobpcg_kwargs["M"] = M
        eigvals, eigvecs = csl.lobpcg(A, X_lobpcg, **lobpcg_kwargs)
        eigvals, eigvecs = _sort_eigpairs(eigvals, eigvecs, xp)
        return SimpleNamespace(values=eigvals[:k], vectors=eigvecs[:, :k])
    except Exception as exc:
        print(f"[LOBPCG fallback] {type(exc).__name__}: {exc}  (k={k}, X={X_lobpcg.shape}, maxiter={_maxiter_lobpcg})")
        fb_iter = int(_cfg_get(cfg, "fallback_maxiter", 3))
        return rand_subspace_rr(
            matvec=matvec,
            size=size,
            top_q=top_q,
            matvec_block=matvec_block,
            block_size=block_size,
            oversample=oversample,
            tol=tol,
            maxiter=fb_iter,
            xp=xp,
            cfg=cfg,
        )


CUSTOM_EIGENSPACE_METHODS = [
    {
        "name": "cupy_eigsh",
        "method": "custom",
        "solver": "eigsh",
        "block_size": None,
        "oversample": OVERSAMPLE_DEFAULT,
        **CUSTOM_EIGSH_CFG,
        "callable": cupy_eigsh,
    },
    {
        "name": "cupy_lobpcg_refine",
        "method": "custom",
        "solver": "lobpcg",
        **CUSTOM_LOBPCG_REFINE_CFG,
        "callable": cupy_lobpcg_refine,
        # Note: CuPy lobpcg is unstable with complex128 / no preconditioner.
        # Falls back to rand_subspace_rr automatically if lobpcg raises.
    },
    {
        "name": "rand_subspace_rr",
        "method": "custom",
        **CUSTOM_RAND_SUBSPACE_RR_CFG,
        "callable": rand_subspace_rr,
    },
    {
        "name": "mixed_subspace_rr",
        "method": "custom",
        **CUSTOM_MIXED_SUBSPACE_RR_CFG,
        "callable": mixed_subspace_rr,
    },
    {
        "name": "poly_filtered_subspace",
        "method": "custom",
        **CUSTOM_POLY_FILTERED_SUBSPACE_CFG,
        "callable": poly_filtered_subspace,
    },
    {
        "name": "mixed_poly_filtered_subspace",
        "method": "custom",
        **CUSTOM_MIXED_POLY_FILTERED_SUBSPACE_CFG,
        "callable": mixed_poly_filtered_subspace,
    },
]

_CUSTOM_EIGENSPACE_STALE_NAMES = {
    "custom_block_lanczos",
    "custom_rand_lanczos",
    "custom_lobpcg_precond",
    "custom_block_power",
    "custom_rand_subspace",
    "custom_mixed_precision_deflation",
    "custom_poly_filter_svd",
    "custom_mixed_poly_filter_svd",
    "cupy_eigsh",
    "cupy_lobpcg_refine",
    "rand_subspace_rr",
    "mixed_subspace_rr",
    "poly_filtered_subspace",
    "mixed_poly_filtered_subspace",
}


def _register_custom_methods():
    existing = {cfg.get("name") for cfg in EIG_METHODS}
    allowed = CUSTOM_METHODS_ENABLED
    if allowed is not None:
        allowed = {name for name in allowed}
    for cfg in CUSTOM_EIGENSPACE_METHODS:
        if cfg["name"] in existing:
            continue
        if allowed is not None and cfg["name"] not in allowed:
            continue
        EIG_METHODS.append(cfg)


if ENABLE_CUSTOM_EIGENSPACE:
    EIG_METHODS[:] = [cfg for cfg in EIG_METHODS if cfg.get("name") not in _CUSTOM_EIGENSPACE_STALE_NAMES]
    _register_custom_methods()


## Step 1: 估计 A 的 top eigenspace（算法步骤 5）

- 这里只做 GPU eigenspace 估计与速度/残差对比（NUFFT 仍在 CPU）
- 配置单元已限定为三种方法：默认 `v3_subspace_iter`、本仓库的 `eigenpro_nystrom`、以及 `cupy_eigsh`（`CUSTOM_EIGSH_CFG`）；`COMBO_ENABLED=False`，其它 custom 未注册
- 其它多精度 / 自定义子空间例程仍保留在下方代码单元，需要时可把 `CUSTOM_METHODS_ENABLED` 改回多选


In [19]:
eig_rows = []
eig_cache = {}
eig_metrics = {}
state_cache = {}

for eps in EPS_LIST:
    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=REG_LAMBDA,
        eps=eps,
        nufft_tol=1e-10,
        l2scaled=L2_SCALED,
    )
    gpu_cfg = GPURunConfig(
        reg_lambda=REG_LAMBDA,
        tol=GPU_TOL,
        maxiter=GPU_MAXITER,
        chunk_size=GPU_CHUNK_SIZE,
        debug_finite_checks=GPU_DEBUG_FINITE_CHECKS,
        backend=BackendConfig(nufft=GPU_NUFFT),
    )

    backend, data_ctx, op_ctx, precompute_time = _build_gpu_context(
        solver,
        x_train,
        y_train,
        gpu_cfg,
    )
    state_cache[eps] = {
        "solver": solver,
        "gpu_cfg": gpu_cfg,
        "backend": backend,
        "data_ctx": data_ctx,
        "op_ctx": op_ctx,
        "time_precompute": precompute_time,
    }

    mtot = int(data_ctx.meta.get("mtot", 0))
    m = int((mtot - 1) // 2) if mtot else np.nan

    for top_q in TOP_Q_LIST:
        top_q = int(top_q)
        if top_q <= 0:
            continue
        for cfg in EIG_METHODS:
            method_name = cfg["name"]
            sur_cfg = cfg.get("surrogate", {}) if cfg.get("method") == "surrogate" else {}
            oversample = int(sur_cfg.get("oversample", cfg.get("oversample", 2)))
            block_size = _resolve_block_size(top_q + 1, sur_cfg or cfg)
            try:
                t_e0 = time.perf_counter()
                eigpairs, block_size, eig_diag = _run_eigenspace(
                    cfg,
                    backend,
                    data_ctx,
                    op_ctx,
                    top_q,
                    solver,
                    gpu_cfg,
                )
                t_e1 = time.perf_counter()

                eigvals_host = _device_to_numpy(backend.xp, eigpairs.values)
                lambda_1 = float(np.real(eigvals_host[0]))
                lambda_q = float(np.real(eigvals_host[top_q - 1]))
                lambda_qp1 = float(
                    np.real(eigvals_host[min(top_q, eigvals_host.size - 1)])
                )

                surrogate_eig_time = float(eig_diag.get("surrogate_time_eigenspace", np.nan))
                surrogate_refine_time = float(eig_diag.get("surrogate_time_refine", np.nan))
                time_eig = float(t_e1 - t_e0)
                if cfg.get("method") == "surrogate":
                    time_eig = surrogate_eig_time + surrogate_refine_time

                row = {
                    "eps": eps,
                    "top_q": top_q,
                    "m": m,
                    "eig_method": method_name,
                    "eig_solver": cfg.get("solver", cfg["method"]),
                    "block_size": block_size,
                    "oversample": oversample,
                    "time_precompute": precompute_time,
                    "time_eigenspace": time_eig,
                    "eig_residual_fro": float(eig_diag.get("residual_fro", np.nan)),
                    "eig_residual_fro_rel": float(eig_diag.get("residual_fro_rel", np.nan)),
                    "lambda_1": lambda_1,
                    "lambda_q": lambda_q,
                    "lambda_q_plus_1": lambda_qp1,
                    "nufft_stage": str(data_ctx.meta.get("nufft_stage", "")),
                    "device_name": str(backend.device_name),
                    "surrogate_tag": eig_diag.get("surrogate_tag", ""),
                    "surrogate_eps": float(eig_diag.get("surrogate_eps", np.nan)),
                    "surrogate_m": float(eig_diag.get("surrogate_m", np.nan)),
                    "surrogate_subsample_n": float(eig_diag.get("surrogate_subsample_n", np.nan)),
                    "surrogate_refine_iter": float(eig_diag.get("surrogate_refine_iter", np.nan)),
                    "surrogate_n_iter": float(eig_diag.get("surrogate_n_iter", np.nan)),
                    "surrogate_time_eigenspace": surrogate_eig_time,
                    "surrogate_time_refine": surrogate_refine_time,
                    "surrogate_precompute": float(eig_diag.get("surrogate_precompute", np.nan)),
                    "status": "ok",
                    "error": "",
                }
                eig_rows.append(row)
                eig_cache[(eps, top_q, method_name)] = eigpairs
                eig_metrics[(eps, top_q, method_name)] = row
            except Exception:
                tb = traceback.format_exc()
                print(f"[EIG ERROR] eps={eps}, top_q={top_q}, method={method_name}")
                print(tb)
                eig_rows.append(
                    {
                        "eps": eps,
                        "top_q": top_q,
                        "m": m,
                        "eig_method": method_name,
                        "eig_solver": cfg.get("solver", cfg["method"]),
                        "block_size": block_size,
                        "oversample": oversample,
                        "time_precompute": precompute_time,
                        "time_eigenspace": np.nan,
                        "eig_residual_fro": np.nan,
                        "eig_residual_fro_rel": np.nan,
                        "lambda_1": np.nan,
                        "lambda_q": np.nan,
                        "lambda_q_plus_1": np.nan,
                        "nufft_stage": str(data_ctx.meta.get("nufft_stage", "")),
                        "device_name": str(backend.device_name),
                        "surrogate_tag": sur_cfg.get("name", ""),
                        "surrogate_eps": float(sur_cfg.get("eps_factor", np.nan)),
                        "surrogate_m": np.nan,
                        "surrogate_subsample_n": float(sur_cfg.get("subsample_frac", np.nan)),
                        "surrogate_refine_iter": float(sur_cfg.get("refine_iter", np.nan)),
                        "surrogate_n_iter": float(sur_cfg.get("sur_iter", np.nan)),
                        "surrogate_time_eigenspace": np.nan,
                        "surrogate_time_refine": np.nan,
                        "surrogate_precompute": np.nan,
                        "status": "error",
                        "error": tb,
                    }
                )


eig_df = pd.DataFrame(eig_rows)
print("Eigenspace rows:", len(eig_df))
eig_df

Eigenspace rows: 3


,eps,top_q,m,eig_method,eig_solver,block_size,oversample,time_precompute,time_eigenspace,eig_residual_fro,...,surrogate_eps,surrogate_m,surrogate_subsample_n,surrogate_refine_iter,surrogate_n_iter,surrogate_time_eigenspace,surrogate_time_refine,surrogate_precompute,status,error
0,0.000001,80,181,subspace_iter,v3_subspace_iter,97,16,0.299167,10.106348,8.473872e+01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,
1,0.000001,80,181,eigenpro_nystrom,v3_eigenpro_nystrom,500,10,0.299167,3.949438,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,
2,0.000001,80,181,cupy_eigsh,eigsh,97,16,0.299167,16.836679,1.454180e-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,


In [20]:
with pd.option_context("display.max_columns", None, "display.width", 200):
    display(eig_df.sort_values(["eps", "top_q", "eig_method"]))

,eps,top_q,m,eig_method,eig_solver,block_size,oversample,time_precompute,time_eigenspace,eig_residual_fro,eig_residual_fro_rel,lambda_1,lambda_q,lambda_q_plus_1,nufft_stage,device_name,surrogate_tag,surrogate_eps,surrogate_m,surrogate_subsample_n,surrogate_refine_iter,surrogate_n_iter,surrogate_time_eigenspace,surrogate_time_refine,surrogate_precompute,status,error
2,0.000001,80,181,cupy_eigsh,eigsh,97,16,0.299167,16.836679,1.454180e-11,1.005962e-15,5.574789e+03,177.595618,177.117218,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,
1,0.000001,80,181,eigenpro_nystrom,v3_eigenpro_nystrom,500,10,0.299167,3.949438,NaN,NaN,1.469045e+06,31282.175666,30169.077113,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,nystrom_s500_q81_oversample10_low0.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,
0,0.000001,80,181,subspace_iter,v3_subspace_iter,97,16,0.299167,10.106348,8.473872e+01,5.862381e-03,5.574789e+03,172.817320,169.622573,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,


## Step 2: 完整算法（步骤 6-8）

- 使用相同 GPU eigenspace 构建预条件 + GPU PCG（NUFFT 仍在 CPU）
- 指标：rmse_test、beta_mse、beta_mae、yhat_mse、yhat_mae


In [21]:
full_rows = []
baseline_cache = {}

if RUN_FULL_SOLVE:
    for eps in EPS_LIST:
        cache = state_cache[eps]
        solver = cache["solver"]
        backend = cache["backend"]
        data_ctx = cache["data_ctx"]
        op_ctx = cache["op_ctx"]
        precompute_time = cache["time_precompute"]

        t_cpu0 = time.perf_counter()
        state_cpu = solver.precompute(x_train, y_train)
        t_cpu1 = time.perf_counter()
        _ = float(t_cpu1 - t_cpu0)

        mtot = int(data_ctx.meta.get("mtot", 0))
        m = int((mtot - 1) // 2) if mtot else np.nan

        for top_q in TOP_Q_LIST:
            top_q = int(top_q)
            if top_q <= 0:
                continue
            baseline_key = (eps, top_q)
            baseline_beta = baseline_cache.get(baseline_key)

            for cfg in EIG_METHODS:
                method_name = cfg["name"]
                key = (eps, top_q, method_name)
                eigpairs = eig_cache.get(key)
                if eigpairs is None:
                    continue
                eig_info = eig_metrics.get(key, {})
                try:
                    t_pb0 = time.perf_counter()
                    precond_data, _mu = _build_precond_gpu(
                        backend,
                        eigpairs.values,
                        eigpairs.vectors,
                        top_q,
                        eig_info,
                    )
                    t_pb1 = time.perf_counter()

                    t_s0 = time.perf_counter()
                    beta_gpu, it, relres, stats = _run_pcg_gpu(
                        backend,
                        data_ctx,
                        op_ctx,
                        precond_data,
                    )
                    t_s1 = time.perf_counter()

                    t_pg0 = time.perf_counter()
                    yhat_gpu_host = _predict_gpu_host(
                        backend,
                        data_ctx,
                        x_test,
                        beta_gpu,
                    )
                    t_pg1 = time.perf_counter()

                    beta_cpu = _device_to_numpy(backend.xp, beta_gpu)
                    t_pc0 = time.perf_counter()
                    yhat_cpu = solver.predict(x_test, beta_cpu, state_cpu)
                    t_pc1 = time.perf_counter()
                    _ = float(t_pc1 - t_pc0)

                    rmse = compute_rmse(yhat_cpu, y_test)
                    yhat_mse, yhat_mae = _mse_mae(yhat_gpu_host, yhat_cpu)

                    beta_mse = np.nan
                    beta_mae = np.nan
                    if method_name == BASELINE_EIG_METHOD:
                        baseline_beta = beta_cpu
                        baseline_cache[baseline_key] = baseline_beta
                    elif baseline_beta is not None:
                        beta_mse, beta_mae = _mse_mae(beta_cpu, baseline_beta)

                    time_eigenspace = float(eig_info.get("time_eigenspace", np.nan))
                    time_solve = float(t_s1 - t_s0)
                    n_precond = int(stats.get("n_precond", 0))
                    t_precond_total = float(stats.get("t_precond_total", np.nan))
                    if n_precond > 0 and not np.isnan(t_precond_total):
                        t_precond_avg = float(t_precond_total / max(n_precond, 1))
                    else:
                        t_precond_avg = np.nan
                    if it and int(it) > 0:
                        t_cg_iter_avg = float(time_solve / max(int(it), 1))
                    else:
                        t_cg_iter_avg = np.nan

                    full_rows.append(
                        {
                            "mode": "gpu_precond",
                            "eps": eps,
                            "top_q": top_q,
                            "eig_method": method_name,
                            "m": m,
                            "rmse_test": float(rmse),
                            "beta_mse": float(beta_mse),
                            "beta_mae": float(beta_mae),
                            "yhat_mse": float(yhat_mse),
                            "yhat_mae": float(yhat_mae),
                            "cg_iters": int(it),
                            "cg_relres": float(relres),
                            "n_matvec": int(stats["n_matvec"]),
                            "t_matvec_avg": float(stats["t_matvec_total"] / max(stats["n_matvec"], 1)),
                            "t_matvec_total": float(stats["t_matvec_total"]),
                            "time_precompute": float(precompute_time),
                            "time_eigenspace": time_eigenspace,
                            "time_precond_build": float(t_pb1 - t_pb0),
                            "eig_residual_fro": float(eig_info.get("eig_residual_fro", np.nan)),
                            "eig_residual_fro_rel": float(eig_info.get("eig_residual_fro_rel", np.nan)),
                            "time_solve": time_solve,
                            "time_predict": float(t_pg1 - t_pg0),
                            "wall_s_total": float(
                                precompute_time
                                + time_eigenspace
                                + (t_pb1 - t_pb0)
                                + time_solve
                                + (t_pg1 - t_pg0)
                            ),
                            "n_precond": n_precond,
                            "t_precond_total": t_precond_total,
                            "t_precond_avg": t_precond_avg,
                            "t_cg_iter_avg": t_cg_iter_avg,
                            "nufft_stage": str(data_ctx.meta.get("nufft_stage", "")),
                            "device_name": str(backend.device_name),
                            "surrogate_tag": eig_info.get("surrogate_tag", ""),
                            "surrogate_eps": float(eig_info.get("surrogate_eps", np.nan)),
                            "surrogate_m": float(eig_info.get("surrogate_m", np.nan)),
                            "surrogate_subsample_n": float(eig_info.get("surrogate_subsample_n", np.nan)),
                            "surrogate_refine_iter": float(eig_info.get("surrogate_refine_iter", np.nan)),
                            "surrogate_n_iter": float(eig_info.get("surrogate_n_iter", np.nan)),
                            "surrogate_time_eigenspace": float(eig_info.get("surrogate_time_eigenspace", np.nan)),
                            "surrogate_time_refine": float(eig_info.get("surrogate_time_refine", np.nan)),
                            "surrogate_precompute": float(eig_info.get("surrogate_precompute", np.nan)),
                            "status": "ok",
                            "error": "",
                        }
                    )
                except Exception:
                    tb = traceback.format_exc()
                    print(f"[SOLVE ERROR] eps={eps}, top_q={top_q}, method={method_name}")
                    print(tb)
                    full_rows.append(
                        {
                            "mode": "gpu_precond",
                            "eps": eps,
                            "top_q": top_q,
                            "eig_method": method_name,
                            "m": m,
                            "rmse_test": np.nan,
                            "beta_mse": np.nan,
                            "beta_mae": np.nan,
                            "yhat_mse": np.nan,
                            "yhat_mae": np.nan,
                            "cg_iters": np.nan,
                            "cg_relres": np.nan,
                            "n_matvec": np.nan,
                            "t_matvec_avg": np.nan,
                            "t_matvec_total": np.nan,
                            "time_precompute": float(precompute_time),
                            "time_eigenspace": float(eig_info.get("time_eigenspace", np.nan)),
                            "time_precond_build": np.nan,
                            "eig_residual_fro": float(eig_info.get("eig_residual_fro", np.nan)),
                            "eig_residual_fro_rel": float(eig_info.get("eig_residual_fro_rel", np.nan)),
                            "time_solve": np.nan,
                            "time_predict": np.nan,
                            "wall_s_total": np.nan,
                            "n_precond": np.nan,
                            "t_precond_total": np.nan,
                            "t_precond_avg": np.nan,
                            "t_cg_iter_avg": np.nan,
                            "nufft_stage": str(data_ctx.meta.get("nufft_stage", "")),
                            "device_name": str(backend.device_name),
                            "surrogate_tag": eig_info.get("surrogate_tag", ""),
                            "surrogate_eps": float(eig_info.get("surrogate_eps", np.nan)),
                            "surrogate_m": float(eig_info.get("surrogate_m", np.nan)),
                            "surrogate_subsample_n": float(eig_info.get("surrogate_subsample_n", np.nan)),
                            "surrogate_refine_iter": float(eig_info.get("surrogate_refine_iter", np.nan)),
                            "surrogate_n_iter": float(eig_info.get("surrogate_n_iter", np.nan)),
                            "surrogate_time_eigenspace": float(eig_info.get("surrogate_time_eigenspace", np.nan)),
                            "surrogate_time_refine": float(eig_info.get("surrogate_time_refine", np.nan)),
                            "surrogate_precompute": float(eig_info.get("surrogate_precompute", np.nan)),
                            "status": "error",
                            "error": tb,
                        }
                    )

full_df = pd.DataFrame(full_rows)
print("Full rows:", len(full_df))
full_df

Full rows: 3


,mode,eps,top_q,eig_method,m,rmse_test,beta_mse,beta_mae,yhat_mse,yhat_mae,...,surrogate_eps,surrogate_m,surrogate_subsample_n,surrogate_refine_iter,surrogate_n_iter,surrogate_time_eigenspace,surrogate_time_refine,surrogate_precompute,status,error
0,gpu_precond,0.000001,80,subspace_iter,181,0.002579,NaN,NaN,7.636683e-23,6.171818e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,
1,gpu_precond,0.000001,80,eigenpro_nystrom,181,0.002578,1.463478e-10,7.749974e-06,7.636689e-23,6.171827e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,
2,gpu_precond,0.000001,80,cupy_eigsh,181,0.002579,1.532612e-12,7.250590e-07,7.636668e-23,6.171817e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,


In [22]:
cols = [
    "mode",
    "eig_method",
    "eps",
    "top_q",
    "m",
    "nufft_stage",
    "device_name",
    "surrogate_tag",
    "surrogate_eps",
    "surrogate_m",
    "surrogate_subsample_n",
    "surrogate_refine_iter",
    "surrogate_n_iter",
    "surrogate_time_eigenspace",
    "surrogate_time_refine",
    "surrogate_precompute",
    "status",
    "rmse_test",
    "beta_mse",
    "beta_mae",
    "yhat_mse",
    "yhat_mae",
    "cg_iters",
    "cg_relres",
    "n_matvec",
    "t_matvec_avg",
    "t_matvec_total",
    "n_precond",
    "t_precond_total",
    "t_precond_avg",
    "time_precompute",
    "time_eigenspace",
    "time_precond_build",
    "eig_residual_fro",
    "eig_residual_fro_rel",
    "time_solve",
    "t_cg_iter_avg",
    "time_predict",
    "wall_s_total",
    "error",
]
for col in cols:
    if col not in full_df.columns:
        full_df[col] = np.nan

with pd.option_context("display.max_columns", None, "display.width", 200):
    display(full_df[cols].sort_values(["eps", "mode", "top_q", "eig_method"]))

,mode,eig_method,eps,top_q,m,nufft_stage,device_name,surrogate_tag,surrogate_eps,surrogate_m,surrogate_subsample_n,surrogate_refine_iter,surrogate_n_iter,surrogate_time_eigenspace,surrogate_time_refine,surrogate_precompute,status,rmse_test,beta_mse,beta_mae,yhat_mse,yhat_mae,cg_iters,cg_relres,n_matvec,t_matvec_avg,t_matvec_total,n_precond,t_precond_total,t_precond_avg,time_precompute,time_eigenspace,time_precond_build,eig_residual_fro,eig_residual_fro_rel,time_solve,t_cg_iter_avg,time_predict,wall_s_total,error
2,gpu_precond,cupy_eigsh,0.000001,80,181,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,0.002579,1.532612e-12,7.250590e-07,7.636668e-23,6.171817e-12,105,8.474014e-07,106,0.004313,0.457168,105,11.280539,0.107434,0.299167,16.836679,0.000422,1.454180e-11,1.005962e-15,11.963804,0.113941,0.017391,29.117463,
1,gpu_precond,eigenpro_nystrom,0.000001,80,181,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,nystrom_s500_q81_oversample10_low0.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,0.002578,1.463478e-10,7.749974e-06,7.636689e-23,6.171827e-12,125,9.753754e-07,126,0.004368,0.550355,125,6.859973,0.054880,0.299167,3.949438,0.000429,NaN,NaN,7.645238,0.061162,0.013637,11.907909,
0,gpu_precond,subspace_iter,0.000001,80,181,cufinufft,NVIDIA GeForce RTX 3050 Laptop GPU,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok,0.002579,NaN,NaN,7.636683e-23,6.171818e-12,113,8.882891e-07,114,0.004333,0.493960,113,12.182331,0.107808,0.299167,10.106348,0.000582,8.473872e+01,5.862381e-03,12.918301,0.114321,0.019552,23.343950,
